# 06. 실습: Deep Agents로 RAG Agent 조립하기

노트북 01~05에서 배운 부품들을 이번에는 **처음부터 한 줄씩 직접 이어 붙여** 완성된 RAG Agent로 조립해 봅니다. 검색 도구, Agent, Skill 연결, 검증, 기록 읽기의 다섯 단계를 차례로 만들고, 여섯 번째 단계로 AutoRAG의 2단계 전체 흐름(`ask_two_phase`)을 얹습니다. 마지막에는 내가 직접 만든 질문 세 개와 바꿔 볼 실험 조건 하나를 골라 전후 비교 실험으로 마무리합니다. 이 실습은 별도 채점을 하지 않으며, 예산 초과나 검증 오류로 실패한 실행 기록도 그 자체로 의미 있는 관찰 결과입니다.

## 학습 목표

- 검색 도구, `create_deep_agent`, Skill 연결, 검증, 기록 읽기의 다섯 단계에 판단 게이트·질문 분해·출처 보강이 결합된 전체 흐름(`ask_two_phase`)을 더해 조립하고, 각 단계가 빠지면 무엇이 깨지는지 설명할 수 있습니다.
- 대표 질문 사례를 실제 모델로 실행하고, 발생한 실패를 검색·범위·Skill·답변·검증·예산·판단 중 어느 층의 실패인지 분류할 수 있습니다.
- 실험 조건을 정확히 하나만 바꿔 같은 질문을 다시 실행하고, 관찰된 결과 차이가 그 조건 때문이라고 말할 수 있는 범위와 한계를 구분할 수 있습니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 지도: 01~05 복습과 여섯 조립 단계 | 4 |
| 2 | 1단계: 검색 도구 (융합 검색 + 재순위) | 6 |
| 3 | 왜 Agent에게 맡기는가 | 2 |
| 4 | 2단계: Agent | 4 |
| 5 | Skill을 잇는 네 가지 인자 | 2 |
| 6 | 3단계: Skill 연결 | 4 |
| 7 | 실패 분류법: 어느 층의 실패인가 | 4 |
| 8 | 4단계: 검증 | 3 |
| 9 | 5단계: 기록 읽기 | 4 |
| 10 | 6단계: 전체 흐름 (`ask_two_phase`) | 6 |
| 11 | 대표 사례 실행, 층 분류 연습 | 6 |
| 12 | 통제 실험 규칙과 1회 실행의 한계 | 3 |
| 13 | 직접 해보기: 내 질문 세 개, 조건 하나 | 9 |
| 14 | 정리와 확인 질문 | 3 |

## 호출 규모

- 이 실습은 실제 Claude 모델과 실제 Decision API만 사용합니다. `Day02_new/.env`에 `ANTHROPIC_API_KEY`와 Decision API 키(`TYPESAFE_API_KEY` 등, `make_decider`가 순서대로 선택)가 모두 설정되어 있어야 합니다. 키가 없으면 첫 코드 셀에서 한국어 안내와 함께 멈추며, 가짜 모델이나 오프라인으로 우회하는 경로는 없습니다.
- 코드 셀 첫 줄 표지: `# [따라하기]` 그대로 실행해 관찰하기, `# [바꿔보기]` 값 하나 바꾸기(**실행 전에 예측 먼저 적기**), `# [깨뜨리기]` 일부러 실패를 일으켜 확인하기, `# [내 것 만들기]` 내 질문과 조건으로 실험하기.
- `announce`는 실행 직전에 예상 Claude 호출 규모를 알려 줍니다. Decision API 호출 횟수는 셀 머리글 주석의 `"판단(Decision API) 약 N회"`를 참고하세요. 보통 한 턴에 2~4회 호출되며 `Budget`의 `max_decision_calls`(기본 6회)가 상한입니다.
- **2026-11-02 실행 기록(실측, 한 대의 기계에서 1회 실행):** 개정한 셀 구성 그대로 `REPEATS = 1`, `MAX_CASES = 3`으로 한 번 실행한 결과 Claude 호출 54회(입력 약 61.7만·출력 약 2.1만 토큰, 약 USD 0.77), Decision API 호출 54회(약 USD 0.0017), 소요 시간 약 260초(약 4.3분)였습니다. 한 대의 기계에서 한 번 돌린 값이므로 모델 응답과 게이트 확률에 따라 달라질 수 있는 관찰이며, 호출 횟수는 추가 확인 게이트가 열려 검증 턴이 도는 질문이 많을수록 늘어납니다. 아래 예상치(약 77회)보다 실측이 적은 것은 `announce`가 셀마다 검증 턴을 포함한 쪽의 호출 수를 알리기 때문이며, 어느 쪽이 맞는 값이라는 뜻이 아니라 추정과 실측이 다르다는 뜻입니다. (이보다 앞선 개정 전 구성의 이전 기록으로는 Claude 호출 48회, Decision API 호출 49회, 약 4.6분, 반복을 2회로 두었던 실행은 Claude 67회가 있었으며, 현재 셀 구성의 실측값이 아닙니다.)
- **현재 판 예상치(추정):** 기본 경로에서 각 셀이 `announce(..., model_calls=N)`로 알리는 값을 모두 더하면 약 77회입니다. 셀별로는 2단계 Agent 3회 + 기준일 바꿔보기 3회 + 3단계 Skill 연결 5회 + 검증기 확인 1회 + 지지 검토 2회 + `ask` 5회 + 예산 부족 `ask` 1회 + `ask_two_phase` 4회 + 대표 사례 12회(`MAX_CASES = 3` 기준 4 x 3) + 다섯 가지 결함 5회 + 내 질문 12회(4 x 3) + 전후 비교 24회(4 x 3 x `REPEATS` 1 x 2)입니다. 이 합계는 `announce`가 알리는 추정치이고 실제 호출 횟수와 다를 수 있습니다. 대표 사례 셀, 내 질문 셀, 전후 비교 셀은 `MAX_CASES`와 `REPEATS`, 질문 수에 비례해 늘어납니다.

## 1. 지도: 여섯 단계

노트북 01~05에서 하나씩 익힌 부품들을 여섯 단계 조립 순서로 요약하면 다음과 같습니다.

| 단계 | 만드는 것 | 어디서 배웠나 |
|---|---|---|
| 1 | 검색 도구 (`@tool`) | 02, 04 |
| 2 | Agent (`create_deep_agent`) | 01, 02 |
| 3 | Skill 연결 (`TrackedSkillsBackend`, `AnswerPayload`) | 03 |
| 4 | 결정적 검증과 의미 검토 (`validate_answer`, `review_support`) | 05 |
| 5 | 실행 기록 읽기와 저장 (`budget.report`, `write_run_record`) | 05 |
| 6 | 2단계 전체 흐름 (`ask_two_phase`: 판단 게이트, 질문 분해, 출처 보강) | 04, 05 |

각 단계를 조립할 때마다 **"만약 이 단계가 없다면 무엇이 깨질까?"** 를 함께 생각해 보세요. 검색 도구가 없으면 문서를 찾을 수 없고, Skill이 없으면 답변 형식이 매번 제멋대로 바뀌며, 검증이 없으면 그럴듯한 거짓말이 그대로 나가고, 실행 기록이 없으면 왜 틀렸는지 사후에 원인을 찾을 수 없습니다.

5단계 끝에서는 `ask()` 함수 한 번이 앞선 다섯 단계를 그대로 묶어 실행한다는 것을 확인하고, 6단계에서는 그 위에 AutoRAG의 판단 게이트와 질문 분해를 얹어 전체 파이프라인을 완성합니다. 먼저 아래 셀을 실행해 두 가지 API 키(`ANTHROPIC_API_KEY`와 Decision API 키)와 기본 모델 객체를 준비합시다.

**Skill이 실제로 발동하는지 시험하는 일은 이 실습의 몫이 아닙니다.** 질문에 맞는 Skill이 선택되는 긍정 질문과, 비슷해 보이지만 선택되면 안 되는 근접 오답(near-miss) 질문으로 발동을 시험하는 실습은 `03-Agent-Skills.ipynb`의 6절과 7절에서 이미 했습니다. 이 실습의 3단계에서는 발동 시험을 다시 하는 대신, 모델이 부른 도구 호출(`search_knowledge`, 그리고 `SKILL.md`를 읽는 `read_file`)과 `loaded_skills` 기록을 눈으로 확인합니다. 또한 6단계 `ask_two_phase`의 빠른 답변(fast) 경로는 코드가 Skill 본문을 프롬프트에 직접 넣어 줍니다. 그래서 그 경로에서 `loaded_skills`에 이름이 남았다는 것은 모델이 스스로 Skill을 불러 발동했다는 뜻이 아니라 코드가 **주입**했다는 뜻입니다(10절 설명과 같습니다).

In [ ]:
from rag_lab.decisions import make_decider
from rag_lab.settings import announce, key_status, make_embeddings, make_model, make_reranker, require_key, ROOT

# [1단계] 키가 설정되어 있는지 이름별로 출력합니다(값은 출력하지 않고 설정됨/미설정만 보여 줍니다)
key_report = key_status()
for key_name, key_state in key_report.items():
    print(f"{key_name:24s} {key_state}")

# [2단계] Anthropic 키와 Decision API 키를 확인하고, 이 실습 전체에서 쓸 모델과 판단 백엔드를 만듭니다
settings = require_key()      # Anthropic API 키가 없으면 한국어 안내와 함께 여기서 멈춥니다.
decider = make_decider()      # Decision API 키가 없으면 여기서 멈춥니다(Claude로 대신하는 경로는 없습니다).

MAX_CASES = 3                 # 11절 대표 사례 셀에서 실제 모델로 실행할 사례 수입니다.
model = make_model()


def report_guess(guess, actual):
    """바꿔보기 셀에서 적은 예측을 실제 결과와 나란히 출력합니다.

    입력: guess는 내가 먼저 적은 예측 값, actual은 실행으로 얻은 실제 값입니다.
    출력: 없음(한 줄을 화면에 출력합니다). 두 값이 같으면 '일치', 다르면 '불일치'라고 적습니다.
    목적: 예측을 먼저 적고 결과와 대조하는 습관을 셀마다 같은 형식으로 남기기 위해서입니다.
    """
    if guess == actual:
        verdict = "일치"
    else:
        verdict = "불일치"
    print(f"예측: {guess} | 실제: {actual} | 일치 여부: {verdict}")


# [3단계] 준비된 설정을 한 줄로 확인합니다
print("MAX_CASES =", MAX_CASES, "| 모델:", model.model, "| 판단(Decision API):", decider.name)

## 2. 1단계: 검색 도구

API 키와 모델, Decision API 백엔드가 모두 준비되었습니다. 이제 첫 번째 조립 단계인 **검색 도구(`@tool`)** 부터 만들어 봅시다.

일반 파이썬 함수 위에 `@tool` 데코레이터를 붙이면 이름, 설명(docstring), 인자 스키마를 갖춘 Agent의 도구가 됩니다. 먼저 세 줄짜리 짧은 사내 안내문으로 가장 작은 검색 도구를 만들어 보겠습니다. 노트북 04에서 배운 `BM25Index`와 `Chunk`를 사용하며, 모델 호출은 없습니다.

In [ ]:
# [따라하기] 세 줄짜리 안내문으로 가장 작은 검색 도구를 만듭니다 (무료, 모델 호출 0회)
from langchain_core.tools import tool
from rag_lab.corpus import Chunk
from rag_lab.retrieval import BM25Index

# [1단계] 안내문 세 문장을 검색 단위(Chunk) 하나씩으로 만들어 BM25 색인에 넣습니다
NOTES = ["사내 식당은 오전 11시 30분에 문을 엽니다.", "도서관 책은 2주 동안 빌릴 수 있습니다.", "통근 버스는 오후 6시에 출발합니다."]
note_chunks = []
for number, sentence in enumerate(NOTES):
    # Chunk(조각 ID, 문서 ID, 제목, 본문, 시작 글자, 끝 글자, 시작 행, 끝 행)
    chunk = Chunk(f"n{number}", "notes", "", sentence, 0, len(sentence), 1, 1)
    note_chunks.append(chunk)
index = BM25Index(note_chunks)


# [2단계] 색인 검색을 @tool로 감싸 Agent가 부를 수 있는 도구로 만듭니다
@tool
def search_notes(query: str, limit: int = 1) -> list:
    """사내 안내문에서 질의와 맞는 문장 번호와 점수를 찾는다."""
    return index.search(query, limit=limit)


# [3단계] 도구를 직접 한 번 불러 결과(문장 번호, BM25 점수)를 확인합니다
print(search_notes.invoke({"query": "식당 언제 열어요?"}))

출력에 나타난 `n0`은 첫 번째 안내문 조각의 ID이고, 옆의 숫자는 BM25 점수입니다. 함수 바로 아래에 적은 설명 글(docstring)이 Agent가 도구를 고를 때 읽는 도구 설명서가 됩니다. 이제 인자 값 하나를 바꿔 봅시다. **실행하기 전에 먼저 예측(`MY_GUESS`)을 적고** 실행해 보세요.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회. 바꿀 값: limit를 1에서 3으로 바꿉니다.
# 먼저 예측해 보세요: limit=3이면 결과가 몇 개 나올까요? 전체 안내문이 세 개이니 3개가 다 나올까요? MY_GUESS에 숫자로 적어 보세요.
MY_GUESS = 3

# [1단계] limit만 3으로 바꿔 같은 질의로 도구를 다시 부릅니다
found = search_notes.invoke({"query": "식당 언제 열어요?", "limit": 3})
print(found)

# [2단계] 실제로 돌아온 결과 개수를 내 예측과 대조합니다
report_guess(MY_GUESS, len(found))
print("안내: BM25는 점수가 0보다 큰 조각만 반환하므로, 질의와 겹치는 단어가 전혀 없는 문장은 후보에 들어오지 않습니다.")

방금 확인했듯 `limit=3`으로 늘려도 질문 단어(`식당`)와 겹치는 문장이 하나뿐이라 결과는 `1`개만 나옵니다.

이제 이 작은 도구 구조를 실제 사내 업무 문서(`data/business/`)로 옮겨 봅시다. 앞선 장난감 예제와 달라지는 핵심은 세 가지입니다:

1. **융합 검색(`mode="fusion"`):** 검색기가 `Retriever(load_corpus(), embeddings=..., mode="fusion")`로 바뀌어 BM25 어휘 점수와 dense 임베딩 점수를 합산합니다.
2. **넓은 후보 수집 + 교차 인코더 재순위(`reranker`):** 검색 도구는 먼저 최대 50개의 후보 조각을 넓게 모은 뒤(`pipeline.retrieve`의 `pool=50`), `make_reranker()`로 만든 로컬 교차 인코더(cross-encoder)가 **질문과 후보 조각을 한 쌍씩 대조해 다시 점수를 매겨** 상위 조각만 근거로 돌려줍니다. (만약 재순위 모델 실행 중 오류가 나면 융합 검색 순서를 그대로 쓰고 기록에 `rerank-failed` 경고를 남깁니다.)
3. **도구 인자에서 범위(`Scope`) 분리:** 도구 인자에는 **고객(`entity`)과 기준일(`as_of`)을 두지 않습니다.** 고객과 기준일은 모델이 마음대로 바꿀 수 없도록 애플리케이션의 `RetrievalContext`가 고정해 들고 있습니다.

In [ ]:
# [따라하기] 실제 사내 문서에 융합 검색 + 재순위 검색 도구를 연결합니다 (무료, 모델 호출 0회)
from datetime import date
from rag_lab.corpus import Scope, load_corpus
from rag_lab.retrieval import Retriever
from rag_lab.evidence import RetrievalContext, make_search_tools

# [1단계] 문서 말뭉치, 임베딩, 재순위기를 준비하고 융합 검색기(search_index)를 만듭니다
embeddings = make_embeddings()
reranker = make_reranker()
corpus = load_corpus()
search_index = Retriever(corpus, embeddings=embeddings, mode="fusion")

# [2단계] 고객(entity)과 기준일(as_of)을 애플리케이션이 고정한 Scope로 묶습니다
scope = Scope(entity="알파", as_of=date(2026, 9, 8))

# [3단계] 검색기, 범위, 재순위기를 RetrievalContext에 담고 검색 도구와 읽기 도구를 만듭니다
context = RetrievalContext(search_index, scope, reranker=reranker)  # 도구가 호출하는 검색은 후보 50개를 모아 재순위합니다.
search_tool, read_tool = make_search_tools(context)

# [4단계] 검색 도구를 한 번 불러 상태, 관련 문서, 충분성 신호를 출력합니다
result = search_tool.invoke({"query": "서비스 크레딧 신청 기한"})
print("status:", result["status"])
print("related_docs:", result["related_docs"])
print("sufficiency(coverage):", result["sufficiency"]["coverage"])

# [5단계] 근거(evidence)마다 ID, 문서, 행 번호, 원문 앞부분 40자를 출력합니다
for item in result["evidence"]:
    quote_head = item["quote"][:40]
    print(f"  {item['evidence_id']}  {item['doc_id']}  L{item['start_line']}-{item['end_line']}  {quote_head!r}")

# [6단계] 도구 인자에 고객/기준일이 없는지, 재순위 경고가 있었는지 확인합니다
print("도구 인자:", list(search_tool.args))
rerank_warnings = []
for event in context.budget.report()["events"]:
    if event["event"] == "retrieval_warning":
        rerank_warnings.append(event["warning"])
print("재순위 경고:", rerank_warnings or "없음")

검색 도구가 돌려준 결과를 살펴봅시다:
- `evidence`: 찾은 원문 인용문(`quote`)과 정확한 좌표(문서 ID, 시작·끝 행 번호).
- `related_docs`: 같은 문서 계열(`family`)에 속하지만 이번 검색 상위권에는 들어오지 않은 관련 문서 ID(개정판이나 부속서를 더 읽어 보라는 힌트).
- `sufficiency`: 질문의 핵심 단어가 근거에 얼마나 포함되었는지 알려 주는 "근거가 부족할 수 있음" 경고 신호.

도구 인자 목록(`search_tool.args`)에 고객(`entity`)이나 기준일(`as_of`)이 빠져 있다는 점과 재순위 경고가 `"없음"`인지도 확인하세요. 참고로 재순위기는 점수가 낮은 하위 후보를 임의로 잘라내지 않으므로, 기본 반환 개수(`limit=6`)를 채우는 과정에서 질문과 무관한 공통 문서(위 출력의 `travel`, 6개 중 3번째 줄)가 근거 목록 안에 함께 딸려 들어올 수 있습니다. 또 위 출력에서 `related_docs`는 빈 목록(`[]`)이므로, 이 질문에서는 더 펼쳐 읽으라는 힌트가 따로 나오지 않았습니다.

이번에는 도구를 일부러 잘못된 조건으로 호출해 세 가지 실패가 어떻게 다르게 처리되는지 확인해 봅시다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회. 세 가지 실패 상황이 서로 어떻게 다르게 처리되는지 비교합니다.
from rag_lab.evidence import Budget, BudgetExceeded

# [1단계] 빈 검색어: 에러 없이 invalid_input 상태값으로 돌아옵니다
print("1) 빈 검색어   :", search_tool.invoke({"query": ""}))

# [2단계] 범위 밖 문서 읽기: 에러 없이 out_of_scope 상태값으로 돌아옵니다(sla-b는 베타 전용 문서)
print("2) 범위 밖 문서:", read_tool.invoke({"doc_id": "sla-b"}))

# [3단계] 검색 예산을 0회로 준 뒤 검색하면 예외(BudgetExceeded)로 실행이 멈춥니다
zero_budget_context = RetrievalContext(search_index, scope, Budget(max_searches=0))
try:
    zero_budget_context.search("서비스 크레딧")
except BudgetExceeded as error:
    print("3) 예산 0회    :", type(error).__name__, error)
print("앞의 두 경우는 에러 없이 상태값(invalid_input, out_of_scope)으로 돌아오고, 세 번째(예산 초과)는 예외로 실행을 즉시 멈춥니다.")

위 세 가지 실패 결과를 비교해 보세요. 빈 검색어(출력의 상태값은 `invalid_input`)나 범위 밖 문서 읽기(`out_of_scope`)는 에이전트가 다른 검색어나 다른 문서로 고쳐서 다음 행동을 이어갈 수 있도록 에러 대신 **상태값**으로 돌려줍니다. 반면 예산 초과(`BudgetExceeded`)는 더 이상 비용을 쓰지 말고 즉시 멈추라는 상한선이므로 **예외(Exception)** 를 발생시켜 실행을 중단시킵니다.

이번에는 검색어는 그대로 둔 채 호출자의 **고객 범위(`entity`)** 만 `"알파"`에서 `"베타"`로 바꾸면 근거 문서 목록이 어떻게 달라지는지 확인해 봅시다. 실행 전에 먼저 예측(`MY_GUESS`)을 적어 보세요.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회. 바꿀 값: 고객(entity)을 "알파"에서 "베타"로 바꿉니다(검색어는 그대로 둡니다).
# 먼저 예측해 보세요: 알파 검색에서 나온 근거 문서 목록과 겹치는 문서가 있을까요? "겹친다" 또는 "겹치지 않는다"로 적어 보세요.
MY_GUESS = "겹치지 않는다"

# [1단계] 고객만 베타로 바꾼 범위와 검색 문맥으로 검색 도구를 새로 만들고 같은 검색어로 부릅니다
beta_scope = Scope(entity="베타", as_of=date(2026, 9, 8))
beta_context = RetrievalContext(search_index, beta_scope, reranker=reranker)
beta_search, _ = make_search_tools(beta_context)
beta_result = beta_search.invoke({"query": "서비스 크레딧 신청 기한"})

# [2단계] 알파 결과(result)와 베타 결과의 근거 문서 ID를 각각 모읍니다
alpha_docs = set()
for item in result["evidence"]:
    alpha_docs.add(item["doc_id"])
beta_docs = set()
for item in beta_result["evidence"]:
    beta_docs.add(item["doc_id"])
print("알파 근거 문서:", sorted(alpha_docs), "| 베타 근거 문서:", sorted(beta_docs))

# [3단계] 두 집합에 겹치는 문서가 있는지 판정해 내 예측과 대조합니다
common_docs = alpha_docs & beta_docs
if common_docs:
    actual_overlap = "겹친다"
else:
    actual_overlap = "겹치지 않는다"
report_guess(MY_GUESS, actual_overlap)

만약 처음 예측과 달리 두 고객의 근거 문서가 겹쳤다면, 어떤 문서가 겹쳤는지 이름을 확인해 보세요. 고객 범위 필터(`Document.applies`)는 해당 고객 전용 문서뿐만 아니라 모든 고객에게 공통으로 적용되는 `공통` 문서(`entity="공통"`)도 함께 허용하기 때문에 출장 규정(`travel`) 문서가 양쪽 모두에 들어왔습니다. 반면 알파 전용 SLA(`sla-a-*`)와 베타 전용 SLA(`sla-b`)는 범위 필터가 정확히 갈라놓았습니다.

## 3. 왜 Agent에게 맡기는가

지금까지 1단계 검색 도구를 만들고 범위 필터와 실패 처리 방식을 확인했습니다. 그런데 검색 도구는 어디까지나 "주어진 검색어로 한 번 찾아 주는 일"만 수행합니다.

실제 업무 질문에 답하려면 어떤 검색어로 몇 번 검색할지, 첫 검색 결과에 뜬 관련 문서(`related_docs`)를 추가로 펼쳐 읽을지 말지를 **상황에 따라 판단하며 반복**해야 합니다. 이 반복 루프를 고정된 `if/else` 코드로 짜지 않고 모델의 판단에 맡기는 틀이 바로 Deep Agents입니다.

물론 반복을 모델에게 맡기면 호출이 길어질수록 비용과 시간이 늘어나므로 **예산(`Budget`)** 이 반드시 함께 붙어야 합니다. Agent에는 모델 호출 횟수, 검색 횟수, 실행 시간 한도가 걸려 있으며, 한도에 닿으면 결과를 조용히 잘라내는 대신 명시적인 예외(`BudgetExceeded`)를 발생시켜 멈춥니다.

## 4. 2단계: Agent

그럼 1단계에서 만든 검색 도구를 `create_deep_agent(model=..., tools=..., system_prompt=...)`에 넘겨 **2단계 Agent**를 조립해 봅시다. 아직 Skill이나 사후 검증은 붙이지 않은 가장 단순한 형태의 Agent입니다. 아래 셀을 실행해 오간 메시지 목록을 한 줄씩 출력하고, 에이전트가 어떤 도구를 어떤 검색어로 불렀는지 읽어 보세요. (모델 호출 약 3회이며, `recursion_limit` 반복 상한을 넘으면 `GraphRecursionError`로 멈춥니다.)

In [ ]:
# [따라하기] 검색 도구만 장착한 2단계 Agent를 실행하고 메시지 흐름을 확인합니다 (유료, 모델 호출 약 3회)
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError

QUESTION = "알파 서비스 크레딧 신청 기한은 언제인가요?"


def run_step2(step_scope):
    """검색 도구만 연결한 2단계 Agent를 지정한 범위로 실행하고, 그 실행의 검색 문맥을 돌려줍니다.

    입력: step_scope는 고객(entity)과 기준일(as_of)을 담은 Scope입니다.
    출력: 이번 실행에서 쓴 RetrievalContext(모은 근거 evidence와 검색 횟수 search_count가 들어 있습니다).
    목적: 같은 질문을 범위만 바꿔 다시 실행하고 모은 근거 문서를 비교하기 위해서입니다.
    예산 초과나 반복 상한 초과로 멈추면 안내를 출력하고, 그때까지 쌓인 검색 문맥을 그대로 돌려줍니다.
    """
    # [1단계] 지정한 고객·기준일 범위(step_scope)로 검색 문맥과 검색 도구를 만듭니다
    step_context = RetrievalContext(search_index, step_scope, reranker=reranker)
    tools = make_search_tools(step_context)
    step2_prompt = (
        f"당신은 문서 근거를 확인하는 한국어 Agent입니다. 고객={step_scope.entity}, 기준일={step_scope.as_of.isoformat()}.\n"
        "search_knowledge로 근거를 찾아 답하세요."
    )

    # [2단계] 검색 도구만 연결한 기본 Deep Agent를 만들고 질문을 실행합니다
    agent = create_deep_agent(model=model, tools=tools, system_prompt=step2_prompt)
    announce("2단계 Agent", model_calls=3)
    try:
        step_result = agent.invoke(
            {"messages": [{"role": "user", "content": QUESTION}]},
            config={"recursion_limit": 20},
        )
    except (BudgetExceeded, GraphRecursionError) as error:
        print("실행이 멈췄습니다:", type(error).__name__, error)
        return step_context

    # [3단계] 주고받은 메시지 목록과 모델이 호출한 검색어(query)를 순서대로 출력합니다
    for message in step_result["messages"]:
        tool_calls = getattr(message, "tool_calls", [])
        call_summaries = []
        for call in tool_calls:
            call_summaries.append((call["name"], call["args"].get("query")))
        message_head = str(message.text)[:90]
        print(f"[{message.type:5s}] {message_head!r} {call_summaries or ''}")
    return step_context


# [4단계] 기본 범위(알파, 9월 8일)로 실행하고 검색 횟수와 모은 근거 문서를 출력합니다
step2_context = run_step2(scope)
step2_doc_ids = set()
for evidence_item in step2_context.evidence.values():
    step2_doc_ids.add(evidence_item.doc_id)
print("검색 횟수:", step2_context.search_count, "| 모은 근거 문서:", sorted(step2_doc_ids))

출력된 `[ai]` 줄 오른쪽의 괄호 안 내용이 모델이 호출한 도구 이름과 검색어(`query`)입니다. 모델이 사용자의 질문을 글자 그대로 검색했는지, 핵심 단어만 추려 다시 썼는지 확인해 보세요.

이제 질문은 그대로 두고 **기준일(`as_of`) 하나만** 바꿔서 다시 실행해 봅니다. 사내 규정 문서에는 개정 이력(`valid_from`, `valid_until`)이 있어서 기준일에 따라 "그날 시점에 유효한 문서 버전"이 달라집니다. 기준일을 9월 8일에서 7월 15일로 바꾸면 에이전트가 모은 근거 문서 목록이 어떻게 달라지는지 확인해 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회. 바꿀 값: 기준일(as_of)을 2026-09-08에서 2026-07-15로 바꿉니다.
# 먼저 예측해 보세요: 기준일을 7월 15일로 바꾸면 모은 근거 문서 목록이 바뀔까요? "바뀐다" 또는 "안 바뀐다"로 적어 보세요.
MY_GUESS = "바뀐다"

# [1단계] 기준일만 7월 15일로 바꾼 범위로 2단계 Agent를 다시 실행합니다(run_step2가 announce를 냅니다)
july_context = run_step2(Scope(entity="알파", as_of=date(2026, 7, 15)))

# [2단계] 9월 실행과 7월 실행이 모은 근거 문서 ID를 각각 정렬된 목록으로 만듭니다
sept_doc_ids = set()
for evidence_item in step2_context.evidence.values():
    sept_doc_ids.add(evidence_item.doc_id)
july_doc_ids = set()
for evidence_item in july_context.evidence.values():
    july_doc_ids.add(evidence_item.doc_id)
sept_docs = sorted(sept_doc_ids)
july_docs = sorted(july_doc_ids)
print("9월 근거 문서:", sept_docs, "| 7월 근거 문서:", july_docs)

# [3단계] 두 목록이 다른지 판정해 내 예측과 대조합니다
if sept_docs != july_docs:
    actual_change = "바뀐다"
else:
    actual_change = "안 바뀐다"
report_guess(MY_GUESS, actual_change)

## 5. Skill을 잇는 네 가지 인자

위 **[바꿔보기]** 셀에서 기준일을 7월 15일로 바꾸자, 8월에 발효된 부속서(`sla-a-amendment`)가 빠지고 7월 시점에 유효한 문서들만 근거로 모인 것을 확인했습니다. 하지만 아직 에이전트의 최종 답변은 자유 형식 텍스트여서 절 제목이나 인용 형식이 일정하지 않습니다.

노트북 03에서 배웠듯 Skill은 `skills/<이름>/SKILL.md` 파일에 담긴 절차 지침서입니다. 이를 2단계 Agent에 연결해 정해진 절차와 형식을 따르는 운영형 에이전트로 만들려면 `create_deep_agent`에 다음 **네 가지 설정**을 더해야 합니다:

- **`backend`와 `skills=["/"]`:** Skill 폴더 위치를 알려 줍니다. 우리가 쓰는 `TrackedSkillsBackend`는 `skills/` 폴더를 읽되, 모델이 **`SKILL.md`를 끝까지 읽었을 때만** `context.loaded_skills`에 읽었다고 기록합니다.
- **`permissions`:** 파일 쓰기(`write`) 권한을 전면 차단(`deny`)합니다. 문서를 조회하는 RAG Agent가 사내 파일이나 지침서를 수정하면 안 되기 때문입니다.
- **`response_format=ProviderStrategy(AnswerPayload)`:** 자유 형식 문자열 대신 정해진 구조(사용한 Skill 이름, 답변 상태, 절 제목, 문장별 인용 ID 목록)로만 최종 답변을 받습니다.
- **`middleware=allowlist_middleware(context.budget)`:** 허용된 도구만 모델에게 보여 주고, 모델·도구 호출 횟수를 예산(`Budget`)에 기록합니다. 시스템 프롬프트도 Skill을 먼저 읽으라고 안내하는 `SYSTEM_PROMPT`로 교체합니다.

## 6. 3단계: Skill 연결

그럼 위 네 가지 설정을 모두 넣어 **3단계 Agent**를 조립하고 실행해 봅시다.

이때 **새로운 실행 턴이므로 `RetrievalContext`를 반드시 새로 만듭니다.** 만약 앞선 2단계에서 썼던 `step2_context`를 그대로 물려받으면 이전 단계의 근거(`evidence`)와 검색 횟수가 남아 있어 뒤이어 할 4단계 사후 검증이 거짓으로 통과해 버립니다. 이 셀의 실행 결과(`result`, `answer`, `context`)는 이어지는 4단계와 5단계에서도 계속 사용합니다. (모델 호출 약 5회)

`05-Production-Agentic-RAG.ipynb`의 11절 '전체 조립'에서는 이미 완성된 조립본을 한 번에 보았습니다. 이 실습에서는 그 완성본을 읽기만 하지 않고, 같은 부품을 **한 단계씩 손으로 이어 붙이며** 각 설정이 무슨 일을 하는지 직접 확인합니다.

In [ ]:
# [따라하기] 3단계: Skill 백엔드, 쓰기 권한 차단, 구조화 응답(AnswerPayload), 허용 목록 미들웨어를 모두 연결합니다 (유료, 모델 호출 약 5회)
from deepagents.middleware.filesystem import FilesystemPermission
from langchain.agents.structured_output import ProviderStrategy
from rag_lab.agent import SYSTEM_PROMPT, TrackedSkillsBackend, allowlist_middleware
from rag_lab.answer import AnswerPayload

# [1단계] 새 턴이므로 앞 단계의 근거를 물려받지 않는 새 RetrievalContext를 만듭니다
context = RetrievalContext(search_index, scope, reranker=reranker)

# [2단계] 네 가지 운영 장치(Skill 백엔드, 쓰기 금지, 구조화 응답, 허용 목록 미들웨어)에 쓸 값을 하나씩 준비합니다
system_prompt = SYSTEM_PROMPT.format(entity=scope.entity, as_of=scope.as_of.isoformat())
skills_backend = TrackedSkillsBackend(ROOT / "skills", context.loaded_skills.add)  # SKILL.md를 끝까지 읽으면 loaded_skills에 기록합니다
deny_write = FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")
tools = make_search_tools(context)

# [3단계] 준비한 값을 모두 넣어 Agent를 조립합니다
agent = create_deep_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    backend=skills_backend,
    skills=["/"],
    permissions=[deny_write],
    response_format=ProviderStrategy(AnswerPayload),
    middleware=allowlist_middleware(context.budget),
)

# [4단계] 에이전트를 실행합니다. 예산 초과나 반복 상한 초과로 멈추면 안내만 출력하고 result는 None으로 둡니다
announce("3단계 Skill 연결", model_calls=5)
result = None
try:
    result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]}, config={"recursion_limit": 40})
except (BudgetExceeded, GraphRecursionError) as error:
    print("실행이 멈췄습니다:", type(error).__name__, error)

# [5단계] 끝까지 읽은 Skill 목록과 구조화된 응답(structured_response)을 확인합니다
if result:
    answer = result["structured_response"]
else:
    answer = None
print("읽은 Skill:", sorted(context.loaded_skills), "| 검색 횟수:", context.search_count)
if answer:
    print("답변 상태:", answer.status, "| 답변 Skill:", answer.skill)
else:
    print("답변 상태:", "답변 없음", "| 답변 Skill:", "-")

출력된 `loaded_skills`에 Skill 이름이 들어 있다면 에이전트가 `read_file`로 해당 `SKILL.md`를 끝까지 읽었다는 뜻이며, 최종 답변의 `answer.skill`은 반드시 이렇게 읽은 Skill 중 하나여야 합니다.

그런데 `skills/` 폴더 안에는 이름이 비슷해 헷갈리기 쉬운 폴더(`grounded-answer` vs `grounded-qa`)가 하나 있습니다. 아래 **[깨뜨리기]** 셀을 실행해 왜 `grounded-answer`는 답변 Skill로 쓸 수 없는지 확인해 봅시다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회. grounded-answer를 답변 Skill로 쓰려 할 때 어떤 오류가 나는지 확인합니다.
from rag_lab.answer import skill_spec

# [1단계] output.json이 없는 grounded-answer를 답변 Skill로 불러 ValueError가 나는지 봅니다
try:
    skill_spec("grounded-answer")
except ValueError as error:
    print("ValueError:", error)

# [2단계] 답변 형식 파일(output.json)이 있는 폴더만 모아 답변 Skill 목록으로 출력합니다
answer_skill_names = []
for output_file in (ROOT / "skills").glob("*/output.json"):
    answer_skill_names.append(output_file.parent.name)
print("답변 Skill 목록:", sorted(answer_skill_names))
print("설명: grounded-answer 폴더에는 output.json이 없습니다. 검색 절차를 안내하는 보조 Skill일 뿐 최종 답변 형식 Skill이 아닙니다.")

## 7. 실패 분류법: 어느 층의 실패인가

방금 본 **[깨뜨리기]** 처럼, 에이전트가 실패하거나 기대와 다른 답을 내놓았을 때 단순히 "모델 성능이 부족하다"고 끝내 버리면 코드의 어디를 고쳐야 할지 알 수 없습니다.

4단계 검증으로 들어가기 전에, 앞으로 우리가 관찰할 모든 실패를 체계적으로 구분하기 위한 **일곱 가지 실패 층** 기준을 먼저 정리해 두겠습니다:

- **검색(`Retrieval`):** 정답을 쓰기에 필요한 문서가 검색된 근거 목록(`evidence`)에 아예 들어오지 않았다. (기록에 `rerank-failed` 경고가 남았다면 이 층을 먼저 의심하는 단서가 됩니다.)
- **범위(`Scope`):** 다른 고객사 문서나 기준일에 만료된 문서를 인용했거나, 반대로 호출자의 범위 설정 때문에 필요한 문서가 사전에 차단되었다.
- **Skill:** 필요한 `SKILL.md`를 끝까지 읽지 않았거나, 질문 유형에 맞지 않는 엉뚱한 Skill을 골랐다.
- **답변(`Answer`):** 충분한 근거가 있는데도 보류(`insufficient`)로 포기했거나, 반대로 근거가 없는데도 지어내어 답(`answered`)했다.
- **검증(`Validation`):** 인용 ID 누락·위조, 인용문에 없는 숫자 사용, 절 제목 불일치 등으로 결정적 검사나 의미 검토를 통과하지 못했다.
- **예산(`Budget`):** 모델 호출·검색 횟수·시간 한도에 걸려 실행이 중간에 멈췄다.
- **판단(`Decision API`):** 2단계 흐름의 게이트가 Decision API 응답을 받지 못해 안전 기본값(`fallback`)을 사용했다. (`gates` 기록에 `fallback`이 남으며, 실행 자체가 에러로 멈추지는 않아 최종 `status`는 정상(`answered`)일 수 있습니다.)

11절에서 사용할 `failure_candidates` 함수가 뽑아 주는 층 후보는 최종 확정 판정이 아니라 **"디버깅할 때 이 층부터 먼저 열어 보라"는 안내 표시**입니다. 예를 들어 "근거가 없는데 답했다"는 현상은 검색이 문서를 못 찾은 것일 수도 있고, 검색은 빈손이었는데 모델이 지어낸 것(답변 층)일 수도 있으며, 문서가 고객 범위 필터에 막힌 것(범위 층)일 수도 있습니다.

## 8. 4단계: 검증

실패를 나누는 일곱 가지 층을 확인했으니, 이제 그중 **검증(`Validation`) 층**을 직접 코드로 붙여 봅시다.

모델이 구조화된 형식(`AnswerPayload`)으로 답변을 돌려주었다고 해서 그 내용이 곧 사실인 것은 아닙니다. 4단계에서는 앞서 3단계에서 얻은 `answer`와 `context`를 `validate_answer` 함수에 넘겨 (1) 인용 ID가 이번 턴에 실제로 모은 근거인지, (2) 원문 해시가 일치하는지, (3) 답변 속 숫자가 인용문 원문에 실제로 등장하는지, (4) 절 제목이 선택한 Skill의 `output.json`과 일치하는지를 파이썬 코드로 검사합니다. (추가 모델 호출은 없습니다.)

In [ ]:
# [따라하기] 4단계: 앞선 3단계에서 얻은 답변(answer)을 validate_answer로 검증하고 마크다운으로 렌더링합니다 (무료, 모델 호출 0회)
from rag_lab.answer import render_answer, validate_answer

if answer is None:
    print("앞 단계에서 답변을 얻지 못했습니다. 검증할 답이 없습니다.")
else:
    # [1단계] 결정적 검증: 인용 ID, 원문 해시, 숫자, 절 제목을 파이썬 코드로 검사합니다
    errors = validate_answer(answer, context, QUESTION)
    print("검증 오류:", errors or "없음")

    # [2단계] 답변을 인용 표기가 붙은 마크다운으로 렌더링해 출력합니다
    print()
    print(render_answer(answer, context))

    # [3단계] 이번 턴에 모은 근거의 ID, 문서, 행 번호, 버전을 한 줄씩 출력합니다
    print()
    for item in context.evidence.values():
        print(f"  {item.evidence_id}  {item.doc_id}  L{item.start_line}-{item.end_line}  v{item.version}")

검증 오류가 `"없음"`으로 나왔더라도 이는 **"답변이 100% 참이다"라는 뜻이 아니라 "`validate_answer`의 규칙 기반 결정적 검사에서 결함이 발견되지 않았다"는 뜻**입니다.

그렇다면 검색 도구를 거치지 않고 모델(`with_structured_output(AnswerPayload)`)이 임의의 인용 ID와 원문에 없는 숫자를 적어 냈을 때 `validate_answer`가 이를 어떻게 잡아내는지 실제 모델을 호출해 확인해 봅시다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 1회 — 검색 도구를 거치지 않고 모델이 임의의 인용 ID와 숫자로 생성한 답변을 검증기가 잡아내는지 확인합니다.
# [1단계] 모델에게 일부러 등록되지 않은 인용 ID와 근거에 없는 숫자를 쓰도록 요청할 프롬프트를 만듭니다
forged_prompt = (
    f"질문: {QUESTION}\n"
    "skill은 'grounded-qa', status는 'answered', 절 제목은 ['한 줄 결론', '근거 요약']으로 작성하되, "
    "문장에 '신청 기한은 45일 이내입니다'라고 쓰고 evidence_ids에는 ['E-unregistered']를 넣으세요."
)

# [2단계] 구조화 출력(AnswerPayload)으로 검색 없이 답변을 생성합니다
announce("검색 근거 없이 구조화 답변(AnswerPayload)을 생성해 결정적 검증기로 검사", model_calls=1)
structured_model = model.with_structured_output(AnswerPayload, method="json_schema")
ungrounded_answer = structured_model.invoke(forged_prompt)
print("모델이 생성한 첫 문장:", ungrounded_answer.sections[0].items[0])

# [3단계] 이 답변을 결정적 검증기에 넣어 검출된 오류를 한 줄씩 출력합니다
print("validate_answer 검출 오류:")
for error in validate_answer(ungrounded_answer, context, QUESTION):
    print(" -", error)

출력의 `validate_answer 검출 오류` 목록에서 이번 턴에 등록되지 않은 인용 ID(`E-unregistered`)와 근거 원문에 없는 숫자(`45`)가 각각 어떤 문구로 잡혔는지 읽어 보세요. 2026-11-02 실행에서는 이 두 가지에 더해 절 제목이 Skill의 `output.json`과 다르다는 오류와, 숫자는 그 문장에 연결한 인용문에 있어야 한다는 오류까지 모두 네 줄이 나왔습니다. 모델이 만드는 가짜 답변의 모양은 실행마다 달라지므로 오류 줄 수는 달라질 수 있지만, 등록되지 않은 ID와 원문에 없는 숫자는 규칙 기반 검사가 모델의 판단과 상관없이 잡는 항목입니다.

하지만 숫자가 없는 문장의 의미 왜곡이나 동문서답은 규칙만으로 잡을 수 없습니다. 그래서 "이 답변 문장이 인용된 근거 원문에 의해 의미적으로 뒷받침되는가?"를 별도의 모델에게 묻는 **의미 검토(`review_support`)**가 존재합니다. 아래 **[바꿔보기]** 셀에서는 앞서 3단계에서 얻은 '신청 기한' 답변(`answer`)을 그대로 둔 채, 질문만 '월간 가용률 99.0% 미만일 때의 크레딧 비율' 질문으로 바꿔서 `review_support`에 넘겨 봅니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 2회. 바꿀 값: review_support에 넘기는 질문을 전혀 다른 사실을 묻는 질문(OTHER_QUESTION)으로 바꿉니다.
# 먼저 예측해 보세요: '신청 기한' 답변을 '월간 가용률 99.0% 미만일 때의 크레딧 비율' 질문에 대해 검토시키면 supported가 True일까요, False일까요?
from rag_lab.answer import review_support

MY_GUESS = False
OTHER_QUESTION = "알파 SLA에서 가용률이 99.0% 미만일 때 서비스 크레딧 비율은 얼마인가요?"
if answer is None:
    print("검토할 답변이 없습니다.")
else:
    # [1단계] 같은 답변을 원래 질문과 다른 질문으로 각각 검토시킵니다(검토마다 모델 호출 1회)
    announce("지지 검토", model_calls=2)
    same = review_support(model, QUESTION, answer, context)
    other = review_support(model, OTHER_QUESTION, answer, context)

    # [2단계] 판정(supported)과 사유(reasons)를 함께 출력하고, 다른 질문의 판정을 내 예측과 대조합니다
    print("원래 질문  supported:", same.supported, same.reasons)
    print("다른 질문  supported:", other.supported, other.reasons)
    report_guess(MY_GUESS, other.supported)

위 셀의 예측과 실제 실행 결과를 비교해 보세요. 2026-11-02 실행에서는 원래 질문과 다른 질문(`OTHER_QUESTION`) 모두 `supported`가 `True`로 나와 `False`를 예측했다면 불일치였습니다. 다만 다른 질문 쪽 사유(`reasons`)의 마지막 줄에는 99.0% 미만의 크레딧 비율 같은 수치를 답변이 단정하지 않았다고 적혀 있어, 이 검토가 본 것은 질문과의 일치가 아니라 답변 진술이 근거 범위 안에 있는지였음을 보여 줍니다. 의미 검토 역시 모델이 수행하므로 실행할 때마다 판정이 달라질 수 있으니, 여러분의 출력에서 두 질문의 `supported`가 같은지 다른지부터 확인하세요. `supported` 불리언 값만 보지 말고 모델이 출력한 사유(`reasons`)를 함께 읽어야 하며, 결정적 검증을 통과한 답변이라도 질문과 어긋난 동문서답일 수 있음을 기억하세요.

## 9. 5단계: 기록 읽기

Agent가 내부에서 어떤 판단과 도구 호출을 거쳤는지는 오직 실행 기록에만 남습니다. `context.budget.report()`를 부르면 호출 횟수(`counts`)와 시간순 사건 목록(`events`)을 얻을 수 있습니다. 여기에 매 실행마다 고유한 `run_id`(`uuid4`)를 붙여 `outputs/runs/<run_id>.json` 파일로 저장하고 다시 읽어 봅시다. 출력에서 호출 횟수, 시간순 이벤트, 검색어, 소요 시간, 그리고 **API 키 문자열(`sk-`)이 기록에 절대 남지 않는다는 점**을 확인하세요.

In [ ]:
# [따라하기] 5단계: 앞선 3단계 실행의 예산·이벤트 기록을 JSON 파일로 저장하고 다시 읽어 확인합니다 (무료, 모델 호출 0회)
import json
from uuid import uuid4
from rag_lab.agent import write_run_record

# [1단계] 예산 기록(호출 횟수와 시간순 사건)을 꺼내 고유한 run_id를 붙인 기록 사전을 만듭니다
trace = context.budget.report()
my_record = {
    "run_id": uuid4().hex,
    "question": QUESTION,
    "trace": trace,
    "loaded_skills": sorted(context.loaded_skills),
}

# [2단계] 기록을 파일로 저장하고, 저장된 파일을 다시 읽어 옵니다
saved_path = write_run_record(my_record)
saved_text = saved_path.read_text(encoding="utf-8")
reread = json.loads(saved_text)

# [3단계] 저장 위치, 호출 횟수, 걸린 시간을 출력합니다
print("기록 파일:", saved_path.relative_to(ROOT))
print("counts:", reread["trace"]["counts"], "| 걸린 시간: %.1f초" % reread["trace"]["elapsed_seconds"])

# [4단계] 시간순 사건 처음 12개를 시각, 종류, 세부 내용(60자까지)으로 출력합니다
for event in reread["trace"]["events"][:12]:
    detail = {}
    for key, value in event.items():
        if key not in ("event", "seconds"):
            detail[key] = value
    print(f"  {event['seconds']:6.2f}s {event['event']:12s} {str(detail)[:60]}")

# [5단계] 기록 파일에 API 키 형식 문자열(sk-)이 남지 않았는지 확인합니다
print("키 형식 문자열이 있는가:", "sk-" in saved_text)

지금까지 우리가 1단계부터 5단계까지 손으로 하나씩 조립한 과정을 `ask()` 함수는 한 번의 호출로 수행합니다(검증 실패 시 최대 1회 재시도 포함). 아래 셀에서 `ask()`를 한 번 실행하고, 반환된 기록 사전(`record`)의 각 키가 우리가 앞선 단계에서 직접 다뤘던 어떤 객체·함수에 대응하는지 주석과 함께 확인해 보세요. (모델 호출 약 5회로 예상하며(2026-11-02 실행에서는 `trace.counts`의 `model`이 3이었습니다), 단일 단계 `ask`에는 Jev 판단 게이트가 없습니다.)

In [ ]:
# [따라하기] 1~5단계 조립 과정을 한 번에 수행하는 ask()를 호출하고 반환 키를 대조합니다 (유료, 모델 호출 약 5회)
from rag_lab.agent import ask

# [1단계] 앞선 단계를 한 번에 묶은 ask()를 실행합니다
announce("ask 한 번", model_calls=5)
record = ask(QUESTION, retriever=search_index, scope=scope, model=model, budget=Budget(), reranker=reranker)

# [2단계] 반환된 기록 사전의 키와 상태, 검증 결과를 출력합니다
print("반환 키:", sorted(record))
print("status:", record["status"], "| validation:", record["validation"], "| errors:", record["errors"] or "없음")

# [3단계] 기록의 각 키가 앞선 단계의 어떤 객체에 대응하는지 나란히 출력합니다
record_doc_ids = []
for evidence_item in record["evidence"]:
    record_doc_ids.append(evidence_item["doc_id"])
markdown_head = record["markdown"][:60].replace("\n", " ")
print("loaded_skills :", record["loaded_skills"], "   <- 3단계의 context.loaded_skills")
print("trace.counts  :", record["trace"]["counts"], "   <- 5단계의 budget.report()")
print("evidence      :", record_doc_ids, "   <- context.evidence")
print("markdown      :", markdown_head, "...   <- 4단계의 render_answer")

또한 `ask()`는 실행 도중 예산 초과나 검증 실패가 발생하더라도 파이썬 예외로 죽지 않고, `status="error"`가 담긴 기록 사전을 돌려주며 파일(`outputs/runs/*.json`)에도 그대로 남깁니다. 아래 **[깨뜨리기]** 셀에서 모델 호출 한도를 일부러 1회(`Budget(max_model_calls=1)`)로 제한해 이를 확인해 봅시다. (실패를 감추기 위해 예산을 올리지 않습니다.)

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 1회. 예산을 일부러 부족하게 주어 status='error' 실패 기록이 파일에 남는지 확인합니다.
# [1단계] 모델 호출 한도를 1회로 제한한 예산으로 ask를 실행합니다(예산을 올려 실패를 감추지 않습니다)
announce("예산 모자란 ask", model_calls=1)
tiny_budget = Budget(max_model_calls=1)
tiny = ask(QUESTION, retriever=search_index, scope=scope, model=model, budget=tiny_budget, max_attempts=1, reranker=reranker)
print("status:", tiny["status"], "| errors:", tiny["errors"])

# [2단계] 실패 기록도 outputs/runs/ 아래 파일로 남았는지 확인합니다
failed_record_path = ROOT / "outputs" / "runs" / f"{tiny['run_id']}.json"
print("실패 기록 파일이 있는가:", failed_record_path.exists())

## 10. 6단계: 전체 흐름 (`ask_two_phase`)

이 과정의 최종 조립 형태는 AutoRAG의 2단계 흐름을 구현한 `ask_two_phase`(`rag_lab/phases.py:125`)입니다. 앞서 본 단일 단계 `ask` 위에 세 가지 Decision API 판단 게이트와 질문 분해, 출처 보강을 얹어 다음 여섯 단계로 돌아갑니다:

1. **분해 게이트(`decompose`):** 복합 질문이어서 검색을 여러 번 나누어 해야 하는지 Decision API가 확률로 판단합니다. 확률이 문턱값(`0.5`) 이상이면 `decompose_question`이 질문을 하위 질의들로 나눕니다(분해 옵션은 기본으로 켜져 있으며 질문마다 게이트가 결정합니다).
2. **미리 검색(`prefetch`):** 나뉜 모든 질의의 검색 후보를 합쳐 원 질문 기준으로 재순위합니다. 이때 출처 보강도(`corroboration`)가 켜져 있어 여러 하위 질의에서 같은 문서 조각이 반복해 등장하면 점수를 보강합니다.
3. **빠른 답변(`fast`):** 에이전트 도구 호출 없이, 미리 찾은 근거와 프롬프트에 넣어 준 Skill 본문만으로 초안 답변을 한 번에 생성합니다. 이때 기록의 `loaded_skills`에는 모델이 `read_file`로 직접 읽은 Skill이 아니라, **코드가 빠른 답변 프롬프트에 직접 넣어 준(적재한) 답변용 Skill 이름들**이 기록됩니다.
4. **인용 관련성 게이트(`citation_relevance`):** 초안 답변이 인용한 각 근거 조각이 질문과 무관한지 Decision API에 묻고, 관련성 확률이 `0.3` 미만인 인용을 답변에서 제거합니다.
5. **추가 확인 게이트(`follow_up`):** 초안 답변에 빠진 내용이 있거나 불확실한 부분이 있는지 Decision API에 묻습니다.
6. **검증 턴(`verification`):** 5번 추가 확인 게이트의 결정이 참(`True`)이거나 초안 답변이 결정적 검증(`_validate`)에 걸렸을 때만 실행됩니다. 이때는 `build_rag_agent`(`rag_lab/phases.py:107`)가 앞선 3단계와 똑같은 구조로 Deep Agent를 조립해 원문을 직접 다시 읽고 답변을 수정합니다.

만약 네트워크 오류 등으로 Decision API가 응답하지 못하면, 각 게이트는 AutoRAG의 안전 기본값(**분해 안 함, 인용 유지, 검증 턴 실행**)을 따르고 기록에 `fallback` 진단을 남깁니다. 게이트가 반환하는 숫자는 보정된 확률이 아니며 문턱값은 코드에 정의된 규칙입니다. 참고로 벤치마크 문서(`docs/retrieval_benchmarks.md`)의 이전 측정에서는 수업용 48문항 중 5문항에서 분해 게이트가 열렸습니다(그 문서의 1회 실행 관찰이며, 이 노트북을 실행한 값이 아닙니다).

In [ ]:
# [따라하기] 6단계: 판단 게이트와 빠른 답변·검증 턴이 결합된 ask_two_phase 전체 흐름을 실행합니다 (유료, 모델 호출 약 3회 + 판단 약 3회)
from rag_lab.phases import ask_two_phase


def show_flow(rec):
    """ask_two_phase 실행 기록에서 질의 목록, 게이트별 확률과 결정, 검증 턴 실행 여부, 호출 수를 요약해 출력합니다.

    입력: rec는 ask_two_phase가 돌려준 실행 기록 사전입니다.
    출력: 없음(여러 줄을 화면에 출력합니다).
    목적: 게이트마다 확률과 문턱값, 결정을 한눈에 읽고 검증 턴이 돌았는지 확인하기 위해서입니다.
    """
    # [1단계] 이번 실행이 검색에 쓴 질의 목록을 출력합니다
    print("  질의:", rec["queries"])

    # [2단계] 게이트마다 확률, 문턱값, 결정을 한 줄씩 출력합니다
    for gate in rec["gates"]:
        if gate["gate"] == "citation_relevance":
            # 인용 관련성 게이트는 인용마다 확률이 있으므로 개수와 최저 확률, 뺀 인용 수를 보여 줍니다
            probabilities = gate["probabilities"]
            lowest = min(probabilities.values(), default=None)
            gate_result = f"인용 {len(probabilities)}개 판단, 최저 P={lowest} -> 뺀 인용 {gate['decision']}"
        else:
            if gate["decision"]:
                decision_text = "참"
            else:
                decision_text = "거짓"
            gate_result = f"P={gate['probability']} (문턱값 {gate['threshold']}) -> {decision_text}"

        # Decision API가 실패해 안전 기본값을 썼다면 그 값도 같이 적습니다
        if gate["fallback"]:
            fallback_note = f"  [대체값: {gate['fallback']}]"
        else:
            fallback_note = ""
        print(f"  게이트 {gate['gate']:20s} {gate_result}{fallback_note}")

    # [3단계] 검증 턴 실행 여부, 상태, 검증 결과, 모델·판단 호출 수, 걸린 시간을 한 줄로 출력합니다
    counts = rec["trace"]["counts"]
    if rec["verification"] == "ran":
        verification_text = "돌림"
    else:
        verification_text = "건너뜀"
    print(
        f"  검증 턴: {verification_text} | status: {rec['status']} | 검증: {rec['validation']} | "
        f"모델 호출 {counts.get('model', 0)} | 판단 호출 {counts.get('decision', 0)} | {rec['trace']['elapsed_seconds']:.1f}초"
    )


# [1단계] 전체 흐름을 한 번 실행합니다
announce("6단계 전체 흐름", model_calls=4)
flow = ask_two_phase(QUESTION, retriever=search_index, scope=scope, model=model, decider=decider, reranker=reranker)

# [2단계] 게이트와 검증 턴 요약, 진단 메시지, 근거 문서를 출력합니다
show_flow(flow)
flow_doc_ids = set()
for evidence_item in flow["evidence"]:
    flow_doc_ids.add(evidence_item["doc_id"])
print("진단:", flow["diagnostics"] or "없음", "| 근거 문서:", sorted(flow_doc_ids))

출력에서 각 게이트의 확률(`P`)과 문턱값, 그리고 최종 결정을 읽어 보세요. `검증 턴`이 `"건너뜀"`이면 도구 없이 만든 빠른 답변이 검증을 통과해 그대로 최종 답변으로 나간 것이고, `"돌림"`이면 `build_rag_agent`로 조립된 에이전트가 등판해 한 번 더 원문을 확인한 것입니다. 같은 질문이라도 실행할 때마다 게이트 확률과 결정이 달라질 수 있습니다(벤치마크 문서의 이전 측정에서도 같은 문항의 추가 확인 확률이 `0.82`와 `0.45`로 갈린 사례가 기록되어 있습니다).

이제 방금 실행한 `flow` 기록 안에 담긴 `phases`(단계별 소요 시간과 오류)와 `fast_answer`(도구 없는 초안) 대 `answer`(최종 답변)를 직접 비교해, 2단계 흐름이 내부에서 어떻게 기록되는지 해부해 봅시다.

In [ ]:
# [따라하기] ask_two_phase가 남긴 단계별(phases) 기록과 초안 대 최종 답변을 해부합니다 (무료, 모델 호출 0회)


def count_items(answer_dict):
    """답변 사전의 모든 절(sections)에 들어 있는 문장(items) 수를 셉니다.

    입력: answer_dict는 답변을 사전으로 바꾼 값입니다(없으면 빈 사전).
    출력: 문장 수(정수).
    목적: 초안(fast_answer)과 최종(answer)의 분량을 같은 방식으로 비교하기 위해서입니다.
    """
    total = 0
    for section in answer_dict.get("sections", []):
        total += len(section.get("items", []))
    return total


# [1단계] 실행된 단계(phases)별 소요 시간과 오류 목록을 확인합니다
print("[1] 실행 단계(flow['phases']):")
for step in flow["phases"]:
    print(f"  - {step['name']:14s} | {step['seconds']:5.2f}초 | 오류: {step['errors'] or '없음'}")

# [2단계] 빠른 답변(fast_answer)과 최종 답변(answer)의 Skill·상태·문장 수를 비교합니다
fast_ans = flow["fast_answer"] or {}
final_ans = flow["answer"] or {}
fast_items = count_items(fast_ans)
final_items = count_items(final_ans)
print("\n[2] 초안(fast_answer) vs 최종(answer) 비교:")
print(f"  - 초안  : skill={fast_ans.get('skill')}, status={fast_ans.get('status')}, 문장 수={fast_items}")
print(f"  - 최종  : skill={final_ans.get('skill')}, status={final_ans.get('status')}, 문장 수={final_items}")
print(f"  - 적재된 Skill 기록(loaded_skills): {flow['loaded_skills']}")

`flow["phases"]` 리스트를 보면 `decompose` → `prefetch` → `fast` → `citations` → `gate`(그리고 필요할 때만 `verification`) 순서로 어느 단계에서 시간이 걸렸고 어떤 검증 오류가 발생했는지 한눈에 추적할 수 있습니다. 또한 검증 턴이 생략(`skipped`)되었을 때는 `loaded_skills`에 빠른 답변 프롬프트에 실린 답변용 Skill 전체가 남고, 검증 턴이 실행(`ran`)되었을 때는 에이전트가 검증 턴에서 `read_file`로 직접 읽은 Skill만 남는다는 점(`rag_lab/phases.py:106`)도 확인할 수 있습니다.

## 11. 대표 사례 실행

이제 `data/cases.jsonl`에 담긴 질문 유형별로 첫 번째 사례를 하나씩 골라(총 `MAX_CASES`개) `ask_two_phase`로 실행해 봅니다. **평가 전용 정답표(`data/gold.jsonl`)는 사례를 고르거나 에이전트를 실행하는 경로에서는 절대 읽지 않고**, 모든 실행이 끝난 뒤 결과 표 옆에 기대 문서와 실패 층 후보를 대조할 때만 `load_gold()`로 읽어 옵니다.

In [ ]:
# [따라하기] 질문 유형별 대표 사례를 실행하고 기대 문서와 비교합니다 (유료, 사례당 모델 호출 약 3회 + 판단 약 3회)
from rag_lab.corpus import load_cases
from rag_lab.evaluation import load_gold

# [1단계] 정답표를 보지 않고 질문 유형(type)별 첫 번째 사례를 MAX_CASES개 고릅니다
cases = load_cases()
representatives = {}
for case in cases:
    representatives.setdefault(case["type"], case)  # 유형마다 처음 나온 사례만 남깁니다
picked = list(representatives.values())[:MAX_CASES]
print(f"유형 {len(representatives)}개 중 {len(picked)}개 실행 (MAX_CASES={MAX_CASES}):", [c["id"] for c in picked])


def scope_of(case):
    """사례 한 건의 고객과 기준일로 검색 범위(Scope)를 만듭니다.

    입력: case는 cases.jsonl의 사례 한 건(entity, as_of 키를 가진 사전)입니다.
    출력: 그 사례의 Scope(entity, as_of).
    목적: 사례마다 호출자 범위를 같은 방식으로 만들기 위해서입니다.
    """
    case_as_of = date.fromisoformat(case["as_of"])
    return Scope(entity=case["entity"], as_of=case_as_of)


def cited_docs(rec):
    """실행 기록의 답변이 인용한 문서 ID를 정렬된 목록으로 돌려줍니다.

    입력: rec는 ask 또는 ask_two_phase가 돌려준 실행 기록 사전입니다.
    출력: 답변이 인용한 근거가 속한 문서 ID 목록(중복 없음, 사전순).
    목적: 답변이 실제로 어떤 문서를 근거로 삼았는지 기대 문서와 비교하기 위해서입니다.
    """
    # 근거 ID -> 문서 ID 대응표
    doc_of_evidence = {}
    for evidence_item in rec["evidence"]:
        doc_of_evidence[evidence_item["evidence_id"]] = evidence_item["doc_id"]

    # 답변의 모든 절, 모든 문장에서 인용한 근거 ID를 모읍니다(답변이 없으면 빈 사전으로 취급)
    answer_dict = rec["answer"] or {}
    cited_ids = []
    for section in answer_dict.get("sections", []):
        for item in section["items"]:
            cited_ids.extend(item["evidence_ids"])

    # 대응표에 있는 근거 ID만 문서 ID로 바꿔 중복 없이 정렬합니다
    doc_ids = set()
    for evidence_id in cited_ids:
        if evidence_id in doc_of_evidence:
            doc_ids.add(doc_of_evidence[evidence_id])
    return sorted(doc_ids)


# [2단계] 고른 대표 사례들을 ask_two_phase로 차례대로 실행합니다
announce("대표 사례 실행", model_calls=4 * len(picked))
results = []  # (사례, 실행 기록) 쌍을 순서대로 모읍니다
for case in picked:
    rec = ask_two_phase(
        case["question"],
        retriever=search_index,
        scope=scope_of(case),
        model=model,
        decider=decider,
        reranker=reranker,
    )
    results.append((case, rec))

# [3단계] 에이전트 실행이 모두 끝난 뒤에만 정답표(gold)를 불러와 인용 문서와 기대 문서를 대조합니다
gold = load_gold()
for case, rec in results:
    expected = gold[case["id"]]
    print(f"{case['id']:28s} {rec['status']:13s} 검증 {rec['validation']:7s} 인용 문서 {cited_docs(rec)} / 기대 문서 {expected['expected_docs']}")
    show_flow(rec)
    if rec["errors"]:
        print("    오류:", rec["errors"])

표의 한 줄이 곧 한 번의 실험 관찰이며, 오류가 난 실패 행도 지우지 않고 그대로 남깁니다. 각 사례의 `follow_up` 확률과 `검증 턴` 줄을 나란히 읽어 보세요. 2026-11-02 실행에서는 첫 사례의 추가 확인 게이트 확률이 `0.23`으로 문턱값(`0.5`)보다 낮아 닫혔고 검증 턴도 `건너뜀`이었으며, 두 번째와 세 번째 사례는 확률이 `0.74`, `0.84`로 열려 검증 턴이 `돌림`이었습니다(1회 실행 관찰). 게이트가 닫혔는데도 검증 턴이 돌았다면 빠른 답변이 결정적 검증(예: `'인용문에 없는 숫자'`)에 걸렸기 때문인지 함께 확인하세요. 같은 사례라도 실행마다 게이트 확률은 달라질 수 있습니다.

이제 바로 아래 셀에서 각 실행 행마다 앞서 배운 **실패 층 후보(`failure_candidates`)** 를 규칙으로 추출해 붙여 봅니다. 다시 강조하지만 이 층 후보는 최종 판정이 아니라 **"디버깅할 때 이 층부터 먼저 확인하라"는 안내 표시**입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회. 위 대표 사례 실행 결과에 규칙 기반 실패 층 후보를 붙입니다.
def failure_candidates(rec, expected_status, expected_docs=()):
    """실행 기록 한 건에서 규칙으로 실패 층 후보를 뽑아 이름 목록으로 돌려줍니다.

    입력: rec는 실행 기록 사전, expected_status는 기대한 상태('answered' 또는 'insufficient'),
          expected_docs는 정답이 들어 있어야 할 문서 ID 목록(없으면 비워 둡니다).
    출력: 실패 층 이름 목록(예: ['범위', 'Skill']). 아무것도 걸리지 않으면 ['후보 없음'].
    목적: 확정 판정이 아니라 '디버깅할 때 이 층부터 먼저 열어 보라'는 안내를 주기 위해서입니다.
    """
    layers = []
    record_scope = Scope(entity=rec["scope"]["entity"], as_of=date.fromisoformat(rec["scope"]["as_of"]))

    # [층 1: 범위] 기대 정답 문서 자체가 호출자 Scope(고객·기준일) 밖이어서 차단된 경우
    blocked_by_scope = False
    for doc in expected_docs:
        if not search_index.documents[doc].applies(record_scope):
            blocked_by_scope = True
    if blocked_by_scope:
        layers.append("범위")
    elif expected_docs:
        # [층 2: 검색] 범위 안 문서인데 검색된 근거 목록(evidence)에 다 들어오지 못한 경우
        retrieved_docs = set()
        for evidence_item in rec["evidence"]:
            retrieved_docs.add(evidence_item["doc_id"])
        if not set(expected_docs) <= retrieved_docs:
            layers.append("검색")

    # [층 3: Skill] SKILL.md를 끝까지 읽지 않았거나 Skill 관련 검증 오류가 난 경우
    skill_error = False
    for message in rec["errors"]:
        if "SKILL.md" in message:
            skill_error = True
    if not rec["loaded_skills"] or skill_error:
        layers.append("Skill")

    # [층 4: 검색(재순위 실패)] 교차 인코더 재순위 중 예외가 나 1차 융합 순서로 폴백한 경우
    rerank_failed = False
    for event in rec.get("trace", {}).get("events", []):
        if event.get("warning", "").startswith("rerank-failed"):
            rerank_failed = True
    if rerank_failed:
        layers.append("검색(재순위 실패)")

    # [층 5: 판단(Decision API 대체값)] Jev 게이트 호출 실패로 안전 기본값(fallback)을 쓴 경우
    used_fallback = False
    for gate in rec.get("gates", []):
        if gate["fallback"]:
            used_fallback = True
    if used_fallback:
        layers.append("판단(Decision API 대체값)")

    # [층 6: 예산 / 실행 / 검증] status가 error일 때 오류 메시지 종류로 세분화
    if rec["status"] == "error":
        other_errors = []  # Skill 오류는 층 3에서 이미 다뤘으므로 뺍니다
        for message in rec["errors"]:
            if "SKILL.md" not in message:
                other_errors.append(message)

        has_budget_error = False
        has_run_error = False
        for message in other_errors:
            if "한도" in message:
                has_budget_error = True
            if message.split(":")[0].endswith("Error"):
                has_run_error = True

        if has_budget_error:
            layers.append("예산")
        elif has_run_error:
            layers.append("실행")
        elif other_errors:
            layers.append("검증")

    # [층 7: 답변] 보류해야 할 질문에 답했거나(과잉 답변), 답할 수 있는데 포기한 경우(과잉 보류)
    if rec["status"] == "answered" and expected_status == "insufficient":
        layers.append("답변(근거 부족인데 답함)")
    if rec["status"] == "insufficient" and expected_status == "answered":
        layers.append("답변(답할 수 있는데 포기)")

    if layers:
        return layers
    return ["후보 없음"]


# 실행한 사례마다 정답표의 기대 상태와 기대 문서로 층 후보를 출력합니다
for case, rec in results:
    expected = gold[case["id"]]
    if expected["abstain"]:
        status = "insufficient"
    else:
        status = "answered"
    print(f"{case['id']:28s} 층 후보 {failure_candidates(rec, status, expected['expected_docs'])}")

만약 위 대표 사례 실행에서 세 문항이 모두 성공해 층 후보가 전부 `['후보 없음']`으로 나왔다면, 실제 장애 상황에서 각 실패 층이 어떻게 잡히는지 더 확인해 볼 필요가 있습니다.

그래서 아래 **[바꿔보기]** 셀에서는 **다섯 가지 실제 결함 조건(고객 범위 오설정, 빈 Skill 폴더, 재순위 모델 경로 오류, Decision API 연결 실패, 호출 예산 초과)** 으로 `ask`와 `ask_two_phase`를 **실제로 실행**해 얻은 진짜 실행 기록(`PRACTICE`)을 수집합니다. 각 결함 조건이 어느 실패 층 후보로 분류될지 `MY_GUESSES` 사전에 먼저 적어 본 뒤 실행해 결과를 비교해 보세요. 참고로 `"다른 고객"` 사례는 고객 범위(`Scope`)가 베타로 묶여 알파 계약서를 읽지 못하고 보류(`insufficient`)로 끝나므로 `['범위', '답변(답할 수 있는데 포기)']` 두 층이 함께 잡힙니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 5회 + 판단 약 4회. 바꿀 값: MY_GUESSES에 각 실제 결함 실행마다 예상되는 실패 층 이름을 먼저 적어 보세요.
from rag_lab.decisions import TypeSafeBackend
from rag_lab.rerank import CrossEncoderReranker

# [1단계] 다섯 가지 실제 결함 조건별로 예상되는 실패 층 이름을 적어 봅니다
# 후보 이름: 범위 | 검색 | Skill | 검색(재순위 실패) | 판단(Decision API 대체값) | 예산 | 실행 | 검증 | 답변(근거 부족인데 답함) | 답변(답할 수 있는데 포기)
MY_GUESSES = {
    "다른 고객": "범위",
    "Skill 누락": "Skill",
    "재순위 실패": "검색(재순위 실패)",
    "판단 대체값": "판단(Decision API 대체값)",
    "호출 초과": "예산",
}

# [2단계] 결함 조건에 쓸 준비물을 만듭니다(Skill이 하나도 없는 빈 폴더)
announce("다섯 가지 결함 조건의 실제 실행 기록 수집", model_calls=5)
empty_skills = ROOT / "outputs" / "empty_skills_practice"
empty_skills.mkdir(parents=True, exist_ok=True)

# [3단계] 결함 1, 고객 범위 오설정: 알파 질문을 베타 범위로 실행합니다
rec_wrong_scope = ask_two_phase(
    QUESTION,
    retriever=search_index,
    scope=Scope("베타", date(2026, 9, 8)),
    model=model,
    decider=decider,
    reranker=reranker,
    verify="never",
)

# [4단계] 결함 2, Skill 누락: 빈 Skill 폴더로 ask를 실행합니다
rec_no_skill = ask(
    QUESTION,
    retriever=search_index,
    scope=scope,
    model=model,
    skills_dir=empty_skills,
    max_attempts=1,
)

# [5단계] 결함 3, 재순위 모델 경로 오류: 존재하지 않는 경로의 재순위기를 씁니다
rec_bad_rerank = ask_two_phase(
    QUESTION,
    retriever=search_index,
    scope=scope,
    model=model,
    decider=decider,
    reranker=CrossEncoderReranker("/nonexistent-local-reranker"),
    verify="never",
)

# [6단계] 결함 4, Decision API 연결 실패: 닿지 않는 주소의 판단 백엔드를 씁니다
rec_bad_decider = ask_two_phase(
    QUESTION,
    retriever=search_index,
    scope=scope,
    model=model,
    decider=TypeSafeBackend("unreachable", base_url="http://127.0.0.1:1"),
    reranker=reranker,
    verify="never",
)

# [7단계] 실행 기록을 (기록, 기대 상태, 기대 문서)로 묶습니다. 결함 5, 호출 초과는 앞선 9절의 tiny 기록을 씁니다
PRACTICE = {
    "다른 고객": (rec_wrong_scope, "answered", ["sla-a-v2", "sla-a-amendment"]),
    "Skill 누락": (rec_no_skill, "answered", ["sla-a-v2", "sla-a-amendment"]),
    "재순위 실패": (rec_bad_rerank, "answered", ["sla-a-v2"]),
    "판단 대체값": (rec_bad_decider, "answered", ["sla-a-v2"]),
    "호출 초과": (tiny, "answered", []),  # 앞서 9절에서 Budget(max_model_calls=1)로 실제 실행한 기록입니다
}

# [8단계] 내 예측(MY_GUESSES)과 실제 실행 기록에 대한 failure_candidates() 진단 결과를 나란히 출력해 비교합니다
for name, (rec, expected_status, expected_docs) in PRACTICE.items():
    guess_text = MY_GUESSES[name] or "(비어 있음)"
    candidates = failure_candidates(rec, expected_status, expected_docs)
    print(f"{name:14s} | 실제 status={rec['status']:12s} | 내 예측: {guess_text:22s} | 진단된 층 후보: {candidates}")

## 12. 통제 실험 규칙과 1회 실행의 한계

설정을 바꿔 가며 RAG 파이프라인의 성능을 비교할 때는 다음 **세 가지 통제 실험 규칙**을 지켜야 합니다:

1. **한 번에 조건 하나만 바꿉니다.** 검색 방식과 재순위 여부를 동시에 바꾸면 결과가 달라졌을 때 어느 쪽 때문인지 알 수 없습니다.
2. **똑같은 질문 사례 집합으로 비교합니다.**
3. **변경 전과 변경 후의 모든 행을 빠짐없이 보여 줍니다.** 결과가 좋아진 행만 골라 남기거나 실패한 행을 지우지 않습니다.

또한 에이전트 실험에서는 **같은 질문을 같은 조건으로 다시 실행해도** 모델이 매번 조금씩 다른 검색어를 쓰거나 게이트 확률이 흔들릴 수 있습니다. 그래서 이어지는 마지막 비교 셀은 변경 전(`전`)과 변경 후(`후`) 설정을 **같은 셀 안에서 번갈아 가며** `REPEATS`회씩 실행합니다. 수업 중 API 비용 부담을 줄이기 위해 기본값은 `REPEATS = 1`로 두었지만, 1회 실행만으로는 결과 차이가 조건 변경의 효과인지 우연한 흔들림(노이즈)인지 구분할 수 없습니다. 여유가 있다면 `REPEATS = 2`로 올려 같은 조건 안에서도 결과가 흔들리는지 직접 확인해 보세요.

- **`REPEATS = 2` 이상일 때 볼 수 있는 것:** 같은 조건 안에서의 반복 간 차이(흔들림 폭)와 조건 변경 전후의 차이 크기를 나란히 비교할 수 있습니다.
- **여전히 볼 수 없는 것:** `REPEATS = 1`로는 반복 간 흔들림 자체를 알 수 없고, `REPEATS = 2`로도 통계적 평균·신뢰 구간·우연일 확률(p-value)까지는 말할 수 없습니다.

## 직접 해보기

아래 두 개의 **[내 것 만들기]** 셀을 차례로 실행합니다. 먼저 채워져 있는 예시 값으로 한 번 실행해 전체 흐름을 확인한 뒤, **내가 직접 만든 질문과 바꿔 보고 싶은 조건**으로 수정해 다시 실행해 보세요.

1. **첫 번째 셀 (내 질문 세 개):**
   - 문서 근거로 **답할 수 있는 질문**(`answered`), 문서에 아예 없는 내용을 묻는 **문서에 없는 질문**(`insufficient`), 알파 고객 계약 범위에서 베타 고객 정보를 묻는 **범위 함정 질문**(`insufficient`) 세 개와 각각의 기대 상태를 적습니다(질문 문자열을 빈 문자열 `""`로 두면 실행에서 제외됩니다).
2. **두 번째 셀 (조건 하나만 바꿔 전후 비교):**
   - 바꿀 수 있는 조건 여섯 가지(`chunker`, `mode`, `k`, `reranker`, `decompose`, `corroboration`) 중 **정확히 하나만** 골라 `MY_CHANGE`에 적습니다. 기준선(`BASE`)은 지금까지 우리가 조립한 설정(문단 청킹, 융합 검색, 재순위 켬, `k=3`, 분해 게이트 켬, 출처 보강도 켬)입니다.
   - 먼저 API 비용이 들지 않는 검색 평가로 노트북 04의 어려운 질문 36개(초안 정답표 기준)에 대한 유형별 `recall`을 측정하고, **놓친 문항은 변경 전과 후 모두 빠짐없이 출력**합니다(호출자 범위 밖에 정답이 있는 문항은 따로 집계해 구분합니다).
   - 이어서 첫 번째 셀에서 만든 내 질문 세 개를 변경 전(`전`)과 변경 후(`후`) 설정으로 `REPEATS`회씩(기본 1회) 번갈아 실행해 비교합니다. 실행 전에 `MY_PREDICTION`(`"나아짐"`, `"같음"`, `"나빠짐"`)을 먼저 적어 보세요.

> **예측할 때 반드시 고려할 구조적 한계:**
> 우리 수업용 말뭉치는 전체 조각이 32개뿐인데 검색 후보 풀(`pool`)은 50개입니다. 따라서 범위 필터를 통과한 모든 조각(예: 알파 기준 15개)이 항상 후보 풀 안에 전부 들어오며, 재순위(`reranker="on"`)가 켜져 있으면 교차 인코더가 그 후보 전체의 순서를 처음부터 다시 줄 세웁니다. 즉, **재순위가 켜진 상태에서는 질문 분해(`decompose`)나 출처 보강도(`corroboration`)만 바꿔서는 검색 순위가 구조적으로 바뀔 수 없습니다.** 출처 보강도의 순수 검색 효과를 관찰하려면 먼저 `BASE`의 `reranker`를 `"off"`로 바꾼 뒤 `corroboration`을 비교해야 합니다(기준선을 바꾼 것이므로 기록해 두세요). 또한 질문 분해(`decompose`)는 모델이 하위 질의를 생성해야 하므로 정적 검색 표에는 반영되지 않고 아래 `ask_two_phase` 전후 비교에서만 효과가 나타납니다.

In [ ]:
# [내 것 만들기] 내 질문 세 개를 ask_two_phase로 실행합니다 (유료, 모델 호출 약 12회 + 판단 약 9회).
# 먼저 예시 그대로 실행해 본 뒤, MY_QUESTIONS를 내가 만든 질문으로 바꿔 보세요(빈 문자열 ""로 두면 실행에서 제외됩니다).
MY_ENTITY = "알파"
MY_AS_OF = "2026-09-08"
MY_QUESTIONS = [
    ("답할 수 있는 질문", "서버가 이상해서 누군가를 깨워야 하는 기준 수치가 뭔가요?", "answered"),
    ("문서에 없는 질문", "알파 고객 전담 매니저의 휴대전화 번호는 무엇인가요?", "insufficient"),
    ("범위 함정 질문", "알파 계약 기준으로 베타 서비스의 장애 응답 시간을 알려주세요.", "insufficient"),
]

# [1단계] 빈 질문을 뺀 실행 대상과 내 검색 범위를 만듭니다
my_items = []
for item in MY_QUESTIONS:
    if item[1].strip():
        my_items.append(item)
my_scope = Scope(entity=MY_ENTITY, as_of=date.fromisoformat(MY_AS_OF))
my_results = []                                  # [(라벨, 질문, 기대 상태, 실행 기록)]을 저장합니다.

# [2단계] 질문이 하나도 없으면 안내만 출력하고, 있으면 질문마다 ask_two_phase를 실행합니다
if not my_items:
    print("질문이 모두 비어 있습니다. MY_QUESTIONS에 질문을 하나 이상 적고 다시 실행하세요.")
    print("예: ('답할 수 있는 질문', '알파 SLA 크레딧은 어떻게 신청하나요?', 'answered')")
else:
    announce("내 질문", model_calls=4 * len(my_items))
    for label, question, expected_status in my_items:
        rec = ask_two_phase(question, retriever=search_index, scope=my_scope, model=model, decider=decider, reranker=reranker)
        my_results.append((label, question, expected_status, rec))

        # 결과 한 줄(기대 상태, 실제 상태, 검증, 인용 문서, 층 후보)과 게이트 요약을 출력합니다
        cited = cited_docs(rec)
        layer_candidates = failure_candidates(rec, expected_status)
        print(
            f"[{label}] 기대 {expected_status} -> status={rec['status']} 검증={rec['validation']} "
            f"인용={cited} 층 후보={layer_candidates}"
        )
        show_flow(rec)
        if rec["errors"]:
            print("    오류:", rec["errors"])

위 셀에서 내 질문 세 개가 기본 설정(`BASE`)에서 어떻게 동작하는지 확인하셨나요? 이제 아래 두 번째 **[내 것 만들기]** 셀에서 조건을 **정확히 하나(`MY_CHANGE`)** 만 바꾸어, 정적 검색 성능(`recall`)과 `ask_two_phase` 전후 결과를 통제 실험으로 비교해 봅시다. 실행하기 전에 `MY_PREDICTION`에 결과를 먼저 예측해 적어 보세요.

In [ ]:
# [내 것 만들기] 조건 하나(MY_CHANGE)만 바꿔 정적 검색(recall)과 에이전트(ask_two_phase) 전후를 비교합니다.
# (유료, 모델 호출 최대 약 24회 + 판단 약 20회, REPEATS=2면 두 배. 바꿀 수 있는 항목: chunker | mode | k | reranker | decompose | corroboration)
from rag_lab.corpus import fixed_chunks, section_chunks
from rag_lab.evaluation import evaluate_retrieval, recall_by_type
from rag_lab.pipeline import retrieve

# [1단계] 실험 조건 설정: 바꿀 조건 하나(MY_CHANGE), 내 예측(MY_PREDICTION), 반복 횟수(REPEATS)
MY_CHANGE = {"reranker": "off"}       # 예: {"decompose": "off"}, {"corroboration": "off"}, {"mode": "bm25"}, {"k": 1}
MY_PREDICTION = "같음"                 # 질문 3개 x REPEATS회에서 '기대 상태 + 검증 통과' 합계가 "나아짐" | "같음" | "나빠짐" 중 무엇일지 먼저 적어 보세요.
REPEATS = 1                            # 전과 후를 각각 몇 번 실행할지 결정합니다. 1회는 노이즈와 효과를 가를 수 없으니 여유가 있으면 2로 바꿔 흔들림을 확인하세요.
BASE = {
    "chunker": "section",
    "mode": "fusion",
    "reranker": "on",
    "k": 3,
    "decompose": "on",
    "corroboration": "on",
}
CHUNKERS = {"section": section_chunks, "fixed": fixed_chunks}


def is_good(expected_status, rec):
    """기대한 답변 상태(answered/insufficient)와 일치하고 결정적 검증도 통과했는지 확인합니다.

    입력: expected_status는 기대 상태, rec는 ask_two_phase 실행 기록입니다.
    출력: 상태가 같고 validation이 'passed'이면 True, 아니면 False.
    목적: 전후 비교에서 '통과 행'을 같은 기준으로 세기 위해서입니다.
    """
    status_matches = rec["status"] == expected_status
    validation_passed = rec["validation"] == "passed"
    return status_matches and validation_passed


def format_recall(value):
    """recall 값을 소수 둘째 자리 문자열로 바꿉니다. 값이 없으면(None) '-'를 돌려줍니다."""
    if value is None:
        return "-"
    return f"{value:.2f}"


# [2단계] 조건 유효성 확인: 바꿀 조건이 정확히 하나이고, 가능한 값이며, 기준선과 다른 값이어야 합니다
valid = {
    "chunker": set(CHUNKERS),
    "mode": {"bm25", "dense", "hybrid", "fusion"},
    "reranker": {"on", "off"},
    "k": {1, 2, 3, 4, 5},
    "decompose": {"on", "off"},
    "corroboration": {"on", "off"},
}
if len(MY_CHANGE) == 1:
    (change_key, change_value), = MY_CHANGE.items()
else:
    change_key, change_value = None, None

if change_key not in valid or change_value not in valid[change_key] or change_value == BASE[change_key]:
    print("MY_CHANGE는 항목이 정확히 하나이고 기준선과 다른 값이어야 합니다.")
    print("  기준선:", BASE)
    print("  바꿀 수 있는 항목과 값:", {k: sorted(v) for k, v in valid.items()})
else:
    # [3단계] 변경 전(BASE)과 변경 후(MY_CHANGE) 설정을 만들고, 각각의 검색기와 재순위기를 조립합니다
    after_config = dict(BASE)
    after_config[change_key] = change_value
    configs = {"전": BASE, "후": after_config}

    def build(config):
        """설정 사전 하나로 검색기(Retriever)와 재순위기(reranker)를 만듭니다. 전과 후가 똑같은 생성 함수를 거칩니다.

        입력: config는 chunker, mode, reranker, corroboration 등을 담은 설정 사전입니다.
        출력: (검색기, 재순위기) 쌍. reranker가 'off'면 재순위기 자리에 None이 들어갑니다.
        """
        if config["mode"] != "bm25":
            config_embeddings = embeddings
        else:
            config_embeddings = None  # BM25 전용 모드에서는 임베딩을 쓰지 않습니다
        index_ = Retriever(
            search_index.documents,
            chunker=CHUNKERS[config["chunker"]],
            embeddings=config_embeddings,
            mode=config["mode"],
            corroboration=config["corroboration"] == "on",
        )
        if config["reranker"] == "on":
            active_reranker = reranker
        else:
            active_reranker = None
        return index_, active_reranker

    built = {}
    for name, config in configs.items():
        built[name] = build(config)

    # [4단계] 어려운 질문 36문항과 초안 정답표를 읽고 질문별 범위를 만듭니다(정적 검색 평가 측정에만 읽습니다)
    hard_cases = load_cases(ROOT / "data/cases_hard.jsonl") + load_cases(ROOT / "data/cases_hard2.jsonl")
    hard_gold = {}
    hard_gold.update(load_gold(ROOT / "data/gold_hard.jsonl"))
    hard_gold.update(load_gold(ROOT / "data/gold_hard2.jsonl"))  # 검수 전 초안입니다.
    hard_scopes = {}
    for case in hard_cases:
        hard_scopes[case["id"]] = scope_of(case)
    orders = {"전": {}, "후": {}}      # 질문마다 매긴 상위 50개 조각 순서를 저장해 전후 순위 변화 비교에 씁니다.

    def measure(name, k):
        """실제 ask가 쓰는 pipeline.retrieve와 같은 경로(pool=50)로 순위를 매겨 문서 recall을 잽니다(분해 질의는 넣지 않습니다).

        입력: name은 '전' 또는 '후', k는 상위 몇 개 문서까지 볼지입니다.
        출력: evaluate_retrieval이 돌려주는 문항별 결과 행 목록(놓친 문서 missing_docs 등을 담습니다).
        """
        index_, ranker = built[name]

        def rank(query, query_scope):
            """질의 하나의 상위 50개 조각 ID 순서를 돌려줍니다. 같은 (질의, 고객, 기준일)은 한 번만 계산합니다."""
            key = (query, query_scope.entity, query_scope.as_of)
            if key not in orders[name]:
                retrieved = retrieve(
                    query,
                    retriever=index_,
                    scope=query_scope,
                    reranker=ranker,
                    pool=50,
                    top_n=50,
                )
                orders[name][key] = retrieved.chunk_ids
            return orders[name][key]

        return evaluate_retrieval(index_, hard_cases, hard_gold, k=k, rank=rank)

    def unreachable(row):
        """정답 문서 중 호출자의 범위(고객, 기준일) 밖에 있는 문서가 하나라도 있는지 알려 줍니다.

        입력: row는 문항별 평가 결과 행(case_id, expected_docs를 가집니다).
        출력: 하나라도 범위 밖이면 True. 그런 문항은 어떤 검색 설정으로도 찾을 수 없습니다.
        """
        case_scope = hard_scopes[row["case_id"]]
        for doc in row["expected_docs"]:
            if not search_index.documents[doc].applies(case_scope):
                return True
        return False

    # [5단계] 정적 검색 성능(recall) 전후 비교 (무료, 어려운 질문 36문항): 설정의 k와 k=1로 각각 측정합니다
    gold_rows = {}
    for name, config in configs.items():
        gold_rows[name] = measure(name, config["k"])
    first_rows = {}
    for name in configs:
        first_rows[name] = measure(name, 1)

    # 유형별 recall 표를 출력합니다
    print("바뀐 조건:", {change_key: (BASE[change_key], change_value)})
    print(f"{'유형':12s} {'전 recall':10s} {'후 recall':10s}")
    after_by_type = {}
    for r in recall_by_type(gold_rows["후"]):
        after_by_type[r["type"]] = r
    for r in recall_by_type(gold_rows["전"]):
        before_val = format_recall(r["recall"])
        after_val = format_recall(after_by_type[r["type"]]["recall"])
        print(f"{r['type']:12s} {before_val:10s} {after_val:10s}")

    # 놓친 행을 전과 후 모두 빠짐없이 출력합니다(범위 밖 정답은 따로 표시합니다)
    for name, rows in gold_rows.items():
        missed = [row for row in rows if row["missing_docs"]]
        out_of_scope_rows = [row for row in missed if unreachable(row)]
        in_scope_missed = len(missed) - len(out_of_scope_rows)
        print(
            f"[{name}] 놓친 행 {len(missed)}개 = 범위 안 {in_scope_missed}개 + 범위 밖 정답 {len(out_of_scope_rows)}개 "
            f"(질문 {len(hard_cases)}개, k={configs[name]['k']})"
        )
        for row in missed:
            if unreachable(row):
                note = "  <- 범위 밖 정답"
            else:
                note = ""
            print(f"     {row['case_id']:30s} 놓친 문서 {row['missing_docs']}{note}")

    # 1위 문서만 볼 때(k=1)의 놓친 행도 전과 후 모두 출력합니다
    for name, rows in first_rows.items():
        missed = [row for row in rows if row["missing_docs"]]
        out_of_scope_count = 0
        for row in missed:
            if unreachable(row):
                out_of_scope_count += 1
        print(f"참고 [{name}] 1위 문서만 볼 때(k=1) 놓친 행 {len(missed)}개 = 범위 안 {len(missed) - out_of_scope_count}개 + 범위 밖 정답 {out_of_scope_count}개")

    # 상위 50개 조각 순서가 전과 후에서 달라진 질문 수를 셉니다
    changed = 0
    for key in orders["전"]:
        if orders["전"][key] != orders["후"][key]:
            changed += 1
    print(f"전과 후에서 상위 50개 조각 순서가 달라진 질문: {changed}/{len(hard_cases)}개")

    if change_key == "decompose":
        print("안내: 분해 질의는 모델이 실시간으로 만들어야 하므로 위 정적 검색 표에는 반영되지 않습니다. 분해의 효과는 아래 ask_two_phase 비교에서 확인하세요.")
    elif change_key == "corroboration":
        print("안내: 재순위가 켜져 있으면 전체 조각(32개)이 모두 후보에 들어가 교차 인코더가 순서를 다시 정합니다. 순서가 같다면 보강도 탓이 아니라 이 구조 때문입니다.")

    # [6단계] 내 질문 세 개에 대한 ask_two_phase 에이전트 전후 비교 (유료 API 호출)
    if change_key == "k":
        print("k는 정적 검색 recall 측정에만 쓰이는 값이라 ask_two_phase 실행 결과는 달라지지 않습니다. 전후 에이전트 비교는 생략합니다.")
        report_guess(MY_PREDICTION, "비교하지 않음")
    elif not my_items:
        print("앞 셀(내 질문)에 유효한 질문이 없어 에이전트 비교를 생략합니다.")
        report_guess(MY_PREDICTION, "비교하지 않음")
    else:
        # 앞 셀의 실행 기록을 재사용하지 않고, 같은 셀 안에서 '전'과 '후'를 번갈아 가며 새로 실행합니다.
        announce(f"전후 ask_two_phase 번갈아 {REPEATS}회씩", model_calls=4 * len(my_items) * REPEATS * 2)
        runs = {"전": [], "후": []}
        for repeat in range(1, REPEATS + 1):
            for name, config in configs.items():
                index_, ranker = built[name]
                round_rows = []
                for _, question, _ in my_items:
                    rec = ask_two_phase(
                        question,
                        retriever=index_,
                        scope=my_scope,
                        model=model,
                        decider=decider,
                        reranker=ranker,
                        budget=Budget(),
                        decompose=config["decompose"] == "on",
                    )
                    round_rows.append(rec)
                runs[name].append(round_rows)

        # 질문마다 전과 후, 반복마다의 결과를 한 줄씩 출력합니다
        print()
        for q, (label, question, expected_status) in enumerate(my_items):
            print(f"[{label}] 기대 상태: {expected_status}")
            for name in ("전", "후"):
                for repeat, rows in enumerate(runs[name], 1):
                    rec = rows[q]
                    if is_good(expected_status, rec):
                        pass_label = "통과"
                    else:
                        pass_label = "미통과"
                    if rec["verification"] == "ran":
                        verify_label = "돌림"
                    else:
                        verify_label = "건너뜀"
                    model_cnt = rec["trace"]["counts"].get("model", 0)
                    dec_cnt = rec["trace"]["counts"].get("decision", 0)
                    elapsed = rec["trace"]["elapsed_seconds"]
                    print(
                        f"   {name}{repeat}: {rec['status']}/{rec['validation']} {pass_label}"
                        f" | 인용 {cited_docs(rec)} | 질의 {len(rec['queries'])} | 검증 턴 {verify_label}"
                        f" | 모델 {model_cnt} 판단 {dec_cnt} | {elapsed:.1f}초"
                    )

        # [7단계] 조건과 반복마다 통과 행 수, 모델 호출 수, 판단 호출 수, 걸린 시간을 모읍니다
        good = {}      # 조건 -> 반복마다 통과한 질문 수
        calls = {}     # 조건 -> 반복마다 모델 호출 합계
        judged = {}    # 조건 -> 반복마다 판단(Decision API) 호출 합계
        secs = {}      # 조건 -> 반복마다 걸린 시간 합계
        for name in runs:
            good[name] = []
            calls[name] = []
            judged[name] = []
            secs[name] = []
            for rows in runs[name]:
                passed_rows = 0
                model_total = 0
                decision_total = 0
                seconds_total = 0
                for item, rec in zip(my_items, rows):
                    if is_good(item[2], rec):
                        passed_rows += 1
                    model_total += rec["trace"]["counts"].get("model", 0)
                    decision_total += rec["trace"]["counts"].get("decision", 0)
                    seconds_total += rec["trace"]["elapsed_seconds"]
                good[name].append(passed_rows)
                calls[name].append(model_total)
                judged[name].append(decision_total)
                secs[name].append(seconds_total)

        print()
        print(f"{'조건':4s} {'반복':4s} {'통과 행':8s} {'모델 호출':10s} {'판단 호출':10s} {'초':8s}")
        for name in runs:
            for repeat in range(REPEATS):
                print(
                    f"{name:4s} {repeat + 1:<4d} {good[name][repeat]}/{len(my_items):<6d} "
                    f"{calls[name][repeat]:<10d} {judged[name][repeat]:<10d} {secs[name][repeat]:<8.1f}"
                )

        # [8단계] 반복 간 흔들림 폭(spread)과 조건 변경 전후 효과(effect)를 계산해 내 예측과 대조합니다
        spread = {}
        for name in runs:
            spread[name] = max(good[name]) - min(good[name])
        effect = sum(good["후"]) - sum(good["전"])
        print(f"같은 조건 안의 반복 차이(통과 행): 전 {spread['전']}, 후 {spread['후']} | 조건 차이(후 합계 - 전 합계): {effect:+d}")

        if REPEATS < 2:
            print("반복이 1회라 같은 조건 안의 흔들림 폭을 알 수 없습니다. 조건 차이가 우연인지 효과인지 가를 수 없으니 여유가 있으면 REPEATS = 2로 바꿔 다시 확인해 보세요.")
        elif abs(effect) <= max(spread.values()):
            print("조건 변경 전후의 차이가 같은 조건 안의 반복 흔들림보다 크지 않습니다. 이 표만으로는 조건 변경 때문이라고 말할 수 없습니다.")
        else:
            print(f"조건 변경 전후 차이가 반복 흔들림보다 큽니다. 다만 이 역시 반복 {REPEATS}회, 질문 3개에 대한 관찰입니다.")

        if effect > 0:
            actual_direction = "나아짐"
        elif effect < 0:
            actual_direction = "나빠짐"
        else:
            actual_direction = "같음"
        report_guess(MY_PREDICTION, actual_direction)

기본 설정(`MY_CHANGE = {"reranker": "off"}`, `REPEATS = 1`)으로 저자가 2026-11-02에 한 번 실행한 결과 해설입니다. 정적 검색 항목은 API 호출이 없는 결정적 값이고, 에이전트 항목은 모델 응답에 따라 달라질 수 있습니다:
- **정적 검색(`recall`) 비교:** 상위 3개 문서를 보는 `k=3` 기준에서는 변경 전후의 `recall`이 같았고 놓친 4개 문항은 모두 호출자 범위 밖에 정답이 있는 문항이었습니다. 하지만 상위 50개 조각의 실제 줄 세우기 순서는 36개 질문 모두에서 달라졌으며, 1위 문서만 보는 `k=1` 기준에서는 놓친 문항이 7개에서 10개로 늘었습니다(늘어난 3개는 모두 범위 안에 정답이 있는 문항입니다).
- **에이전트(`ask_two_phase`) 비교:** 내 질문 3개는 변경 전과 후 모두 통과해 조건 차이(`effect`)가 `0`으로 나왔지만, 반복 횟수가 1회뿐이어서 이 `0`이 진짜 동등함인지 우연인지 해석할 수는 없습니다. 이번 실행에서는 총 6번의 에이전트 실행 모두 질의가 1개였고, 차이는 '범위 함정 질문'의 인용 문서가 변경 전(`전1`)에는 `sla-a-amendment`와 `sla-a-v2` 두 개, 변경 후(`후1`)에는 `sla-a-v2` 하나였다는 점뿐입니다. 이런 한 건의 차이는 재순위기를 끈 영향인지 모델 응답의 흔들림인지 한 번의 실행으로 가를 수 없습니다. 여러분의 실행에서는 분해 게이트가 열려 질의 수가 늘어난 행이 있는지, 있다면 그것이 한 번뿐인 확률 흔들림인지 먼저 따져 보세요.
- 참고로 재순위기를 켠 상태(`reranker: "on"`) 그대로 두고 `corroboration`만 `"off"`로 바꿨을 때는 상위 50개 순서가 바뀐 질문이 `0/36`개였습니다(API 호출 없이 별도 확인).

## 확인 질문

1. 2절 마지막 **[바꿔보기]** 셀 출력에서 알파 고객과 베타 고객의 근거 문서 목록 차이와 양쪽에 공통으로 들어간 문서를 확인해 보세요. 만약 검색 도구에 고객(`entity`)이 인자로 열려 있었다면, 모델이 한 번의 실행에서 어떤 잘못된 조회를 할 수 있었을지 구체적인 질문 예시 하나로 적어 보세요.
2. 8절 **[깨뜨리기]** 셀의 `'현재 턴에 없는 인용 ID'` 오류는 어느 객체의 어느 목록과 대조해서 판정되나요? 만약 6절(3단계)에서 새로운 `RetrievalContext`를 만들지 않고 이전 단계의 객체를 그대로 썼다면, 앞 단계의 인용 ID를 가져다 쓴 답변을 이 검증기는 어떻게 처리하게 되나요?
3. 6절 **[깨뜨리기]** 셀에서 발생한 `ValueError`는 7절에서 배운 일곱 가지 실패 층 중 어디에 속하나요? 이는 `loaded_skills`에 없는 Skill 이름을 답변의 `skill` 칸에 적어 낸 경우와 같은 층인가요?
4. 9절 **[깨뜨리기]** 셀 출력을 바탕으로, 만약 같은 질문을 다섯 번 실행해 그중 두 번이 예산 초과(`status="error"`)로 멈췄다면, 실패 기록을 지운 표와 그대로 남긴 표의 성공률은 각각 몇 퍼센트인가요? 두 표 중 어느 쪽이 실제 운영 환경을 반영하나요?
5. 11절 실패 층 후보 표에서 만약 어떤 실행 기록에 `rerank-failed` 경고가 들어 있었다면 그 행의 층 후보는 어떻게 바뀌나요? 그때 검색 도구가 돌려준 문서 순서는 어떤 순서이며, 노트북 04의 어느 표와 비교해야 하나요?
6. 10절 출력의 세 가지 게이트 줄을 보고, 만약 네트워크 장애로 Decision API 호출이 세 번 모두 실패했다면 각 게이트의 결정과 `검증 턴` 실행 여부가 어떻게 달라질지 예측해 보세요. 또한 그 실패 흔적은 반환된 실행 기록(`flow`)의 어느 필드에 남나요?
7. '직접 해보기' 비교 셀에서 재순위(`reranker="on"`)는 그대로 켜 둔 채 출처 보강도(`corroboration`)만 `"off"`로 바꾸면 `전 recall`과 `후 recall`이 같을지 다를지, 노트북 04 출력의 `범위가 허용한 조각` 수(15개)와 후보 풀 상한(`pool=50`)을 근거로 설명해 보세요. 만약 말뭉치 크기가 어떤 조건이면 이 예측이 달라지나요?
8. '직접 해보기' 비교 셀의 마지막 출력 줄(`같은 조건 안의 반복 차이`와 `조건 차이`)을 보고, 본인 출력에서 근거 있게 말할 수 있는 문장 하나와 말할 수 없는 문장 하나를 적어 보세요. 만약 `REPEATS`를 `2`로 올리면 새롭게 확인해 말할 수 있게 되는 것은 무엇인가요?

## 정리

- **다섯 단계 수동 조립과 여섯 번째 전체 흐름:** 검색 도구(`@tool`) → 에이전트(`create_deep_agent`) → Skill 연결(`TrackedSkillsBackend`, `AnswerPayload`) → 결정적 검증과 의미 검토(`validate_answer`, `review_support`) → 실행 기록(`budget.report`, `write_run_record`)의 다섯 단계를 직접 이어 붙였고, 그 위에 AutoRAG의 2단계 흐름(`ask_two_phase`: 분해 게이트 → 미리 검색 → 빠른 답변 → 인용 관련성 게이트 → 추가 확인 게이트 → 검증 턴)을 얹었습니다.
- **일곱 층 실패 분류:** 에이전트가 실패했을 때는 막연히 모델을 탓하지 않고 검색·범위·Skill·답변·검증·예산·판단(Decision API) 중 어느 층의 문제인지 먼저 가려낸 뒤 수정합니다. 판단 게이트가 API 실패로 안전 기본값을 쓴 흔적은 `gates`의 `fallback`과 `diagnostics`에 남습니다.
- **통제 실험의 세 원칙:** 실험 조건을 비교할 때는 한 번에 조건 하나만 바꾸고, 같은 사례 집합으로 측정하며, 변경 전후의 모든 행(실패 및 놓친 문항 포함)을 빠짐없이 기록합니다.
- **관찰과 통계의 구분:** 질문 3개, 반복 1회(기본값) 실행은 파이프라인 동작을 눈으로 확인하는 '관찰'일 뿐 통계적 증명이 아니며, 조건 변경의 효과와 모델의 우연한 흔들림(노이즈)을 가르려면 반복 실행과 충분한 문항 수가 필요합니다.